In [0]:
from pyspark.sql import functions as F

generation_id = "8573dcf32f2ed486"
profile = "medium"
mapping_version = "v1"


def write_generation(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .option(
            "replaceWhere",
            f"_generation_id = '{generation_id}'",
        )
        .saveAsTable(table_name)
    )


def normalized_value(column_name):
    return F.lower(
        F.regexp_replace(
            F.trim(F.col(column_name)),
            r"[^A-Za-z0-9]+",
            "",
        )
    )


def canonical_id(prefix, entity_type, source_id_column):
    digest = F.sha2(
        F.concat_ws(
            "||",
            F.lit(entity_type),
            F.lit("architecture"),
            F.col(source_id_column),
        ),
        256,
    )

    return F.concat(
        F.lit(prefix),
        F.upper(F.substring(digest, 1, 16)),
    )

In [0]:
silver_ecus_df = (
    spark.table("ontologysim.silver.architecture_ecus")
    .filter(F.col("_generation_id") == generation_id)
)

silver_components_df = (
    spark.table("ontologysim.silver.architecture_components")
    .filter(F.col("_generation_id") == generation_id)
)

print("Silver ECUs:", silver_ecus_df.count())
print("Silver Components:", silver_components_df.count())

Silver ECUs: 3
Silver Components: 1600


In [0]:
ecu_null_ids = (
    silver_ecus_df
    .filter(F.col("ecu_id").isNull() | (F.trim("ecu_id") == ""))
    .count()
)

ecu_duplicate_ids = (
    silver_ecus_df
    .groupBy("ecu_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

component_null_ids = (
    silver_components_df
    .filter(
        F.col("component_id").isNull()
        | (F.trim("component_id") == "")
    )
    .count()
)

component_duplicate_ids = (
    silver_components_df
    .groupBy("component_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("ECU null IDs:", ecu_null_ids)
print("ECU duplicate IDs:", ecu_duplicate_ids)
print("Component null IDs:", component_null_ids)
print("Component duplicate IDs:", component_duplicate_ids)

if (
    ecu_null_ids
    or ecu_duplicate_ids
    or component_null_ids
    or component_duplicate_ids
):
    raise ValueError(
        "Architecture authoritative identity validation failed"
    )

print("Authoritative Architecture IDs passed")

ECU null IDs: 0
ECU duplicate IDs: 0
Component null IDs: 0
Component duplicate IDs: 0
Authoritative Architecture IDs passed


In [0]:
canonical_ecus_df = (
    silver_ecus_df
    .select(
        canonical_id(
            "CAN-ECU-",
            "ECU",
            "ecu_id",
        ).alias("canonical_id"),

        F.trim("ecu_id").alias("source_ecu_id"),
        F.trim("ecu_name").alias("ecu_name"),
        F.trim("role").alias("role"),

        F.lit("ARCHITECTURE").alias(
            "authoritative_source"
        ),
        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        "_source_file",
        "_generation_id",
        "_profile",
    )
    .withColumn(
        "_canonical_transformed_at",
        F.current_timestamp(),
    )
)

display(canonical_ecus_df)

canonical_id,source_ecu_id,ecu_name,role,authoritative_source,mapping_version,_source_file,_generation_id,_profile,_canonical_transformed_at
CAN-ECU-B23A779303D7B8A0,ECU-HCP3,HCP3,Infotainment and user-facing connectivity,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:43.970Z
CAN-ECU-24DD72AFD1D44768,ECU-HCP5,HCP5,Vehicle-state coordination and energy-related functions,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:43.970Z
CAN-ECU-6714038EEAC88885,ECU-CONMOD,ConMod,External connectivity and telematics,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:43.970Z


In [0]:
ecu_lookup_df = (
    canonical_ecus_df
    .select(
        F.col("source_ecu_id").alias(
            "lookup_source_ecu_id"
        ),
        F.col("canonical_id").alias(
            "canonical_ecu_id"
        ),
    )
)

canonical_components_df = (
    silver_components_df.alias("component")
    .join(
        ecu_lookup_df.alias("ecu"),
        F.col("component.ecu_id")
        == F.col("ecu.lookup_source_ecu_id"),
        "left",
    )
    .select(
        canonical_id(
            "CAN-CMP-",
            "COMPONENT",
            "component.component_id",
        ).alias("canonical_id"),

        F.trim(
            "component.component_id"
        ).alias("source_component_id"),

        F.trim(
            "component.component_name"
        ).alias("component_name"),

        F.trim(
            "component.component_type"
        ).alias("component_type"),

        F.trim(
            "component.ecu_id"
        ).alias("source_ecu_id"),

        F.col("ecu.canonical_ecu_id"),

        F.trim(
            "component.responsibility"
        ).alias("responsibility"),

        F.trim(
            "component.version"
        ).alias("version"),

        F.trim(
            "component.status"
        ).alias("status"),

        F.trim(
            "component.owner_team"
        ).alias("owner_team"),

        F.trim(
            "component.parent_component_id"
        ).alias("source_parent_component_id"),

        F.lit("ARCHITECTURE").alias(
            "authoritative_source"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        F.col("component._source_file").alias(
            "_source_file"
        ),
        F.col("component._generation_id").alias(
            "_generation_id"
        ),
        F.col("component._profile").alias(
            "_profile"
        ),
    )
    .withColumn(
        "_canonical_transformed_at",
        F.current_timestamp(),
    )
)

display(canonical_components_df)

canonical_id,source_component_id,component_name,component_type,source_ecu_id,canonical_ecu_id,responsibility,version,status,owner_team,source_parent_component_id,authoritative_source,mapping_version,_source_file,_generation_id,_profile,_canonical_transformed_at
CAN-CMP-DB1BD195E0FECE45,SWC-HCP3-BT-001,BluetoothConnectionManager,SOFTWARECOMPONENT,ECU-HCP3,CAN-ECU-B23A779303D7B8A0,Controls Bluetooth connection and reconnection behavior.,3.4.0,ACTIVE,Connectivity,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-B1CF366DEC7F8B20,SWC-HCP3-CONN-002,ConnectivityStatusService,SERVICE,ECU-HCP3,CAN-ECU-B23A779303D7B8A0,Provides user-facing connectivity state.,3.4.0,ACTIVE,Connectivity,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-9F79C1A5AEC97E29,SWC-HCP3-MEDIA-003,MediaSessionManager,SOFTWARECOMPONENT,ECU-HCP3,CAN-ECU-B23A779303D7B8A0,Coordinates Bluetooth media sessions.,3.4.0,ACTIVE,Infotainment,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-EAD09678DB459E65,SWC-HCP3-PROFILE-004,UserProfileManager,SOFTWARECOMPONENT,ECU-HCP3,CAN-ECU-B23A779303D7B8A0,Maintains user and paired-device context.,3.4.0,ACTIVE,Infotainment,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-A4C4A48E5AA09FDE,SWC-HCP5-PWR-001,VehiclePowerStateManager,SOFTWARECOMPONENT,ECU-HCP5,CAN-ECU-24DD72AFD1D44768,Maintains vehicle power state.,3.4.0,ACTIVE,VehicleState,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-F8504D0ECACF284A,SWC-HCP5-WAKE-002,WakeSleepCoordinator,SOFTWARECOMPONENT,ECU-HCP5,CAN-ECU-24DD72AFD1D44768,Coordinates vehicle wake and sleep transitions.,3.4.0,ACTIVE,VehicleState,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-1C399A982DF152F5,SWC-HCP5-STATE-003,VehicleStateDistributor,SERVICE,ECU-HCP5,CAN-ECU-24DD72AFD1D44768,Distributes vehicle-state changes to dependent ECUs.,3.4.0,ACTIVE,VehicleState,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-C2F9CF14E6DCD7B1,SWC-HCP5-ENERGY-004,EnergyManagementService,SERVICE,ECU-HCP5,CAN-ECU-24DD72AFD1D44768,Provides energy-management context.,3.4.0,ACTIVE,EnergyManagement,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-99EB2004479617C7,SRV-CON-NET-001,ConnectivityService,SERVICE,ECU-CONMOD,CAN-ECU-6714038EEAC88885,Provides external network connectivity state.,3.4.0,ACTIVE,Connectivity,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z
CAN-CMP-99C1D69D5AD519AB,SWC-CON-CELL-002,CellularManager,SOFTWARECOMPONENT,ECU-CONMOD,CAN-ECU-6714038EEAC88885,Controls cellular network registration.,3.4.0,ACTIVE,Connectivity,,ARCHITECTURE,v1,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml,8573dcf32f2ed486,medium,2026-10-14T06:23:53.695Z


In [0]:
ecu_identity_map_df = (
    canonical_ecus_df
    .select(
        "canonical_id",

        F.lit("ECU").alias(
            "canonical_entity_type"
        ),

        F.lit("architecture").alias(
            "source_family"
        ),

        F.lit("ECU").alias(
            "source_entity_type"
        ),

        F.col("source_ecu_id").alias(
            "source_id"
        ),

        F.col("ecu_name").alias(
            "source_name"
        ),

        normalized_value("ecu_name").alias(
            "normalized_source_value"
        ),

        F.lit("MATCHED").alias(
            "mapping_status"
        ),

        F.lit("DIRECT_ID").alias(
            "mapping_method"
        ),

        F.lit(1.0).alias(
            "mapping_confidence"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        "_generation_id",
        "_source_file",
    )
)

component_identity_map_df = (
    canonical_components_df
    .select(
        "canonical_id",

        F.lit("COMPONENT").alias(
            "canonical_entity_type"
        ),

        F.lit("architecture").alias(
            "source_family"
        ),

        F.lit("COMPONENT").alias(
            "source_entity_type"
        ),

        F.col("source_component_id").alias(
            "source_id"
        ),

        F.col("component_name").alias(
            "source_name"
        ),

        normalized_value("component_name").alias(
            "normalized_source_value"
        ),

        F.lit("MATCHED").alias(
            "mapping_status"
        ),

        F.lit("DIRECT_ID").alias(
            "mapping_method"
        ),

        F.lit(1.0).alias(
            "mapping_confidence"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        "_generation_id",
        "_source_file",
    )
)

identity_map_df = (
    ecu_identity_map_df
    .unionByName(component_identity_map_df)
)

display(identity_map_df)

canonical_id,canonical_entity_type,source_family,source_entity_type,source_id,source_name,normalized_source_value,mapping_status,mapping_method,mapping_confidence,mapping_version,_generation_id,_source_file
CAN-ECU-B23A779303D7B8A0,ECU,architecture,ECU,ECU-HCP3,HCP3,hcp3,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-ECU-24DD72AFD1D44768,ECU,architecture,ECU,ECU-HCP5,HCP5,hcp5,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-ECU-6714038EEAC88885,ECU,architecture,ECU,ECU-CONMOD,ConMod,conmod,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-DB1BD195E0FECE45,COMPONENT,architecture,COMPONENT,SWC-HCP3-BT-001,BluetoothConnectionManager,bluetoothconnectionmanager,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-B1CF366DEC7F8B20,COMPONENT,architecture,COMPONENT,SWC-HCP3-CONN-002,ConnectivityStatusService,connectivitystatusservice,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-9F79C1A5AEC97E29,COMPONENT,architecture,COMPONENT,SWC-HCP3-MEDIA-003,MediaSessionManager,mediasessionmanager,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-EAD09678DB459E65,COMPONENT,architecture,COMPONENT,SWC-HCP3-PROFILE-004,UserProfileManager,userprofilemanager,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-A4C4A48E5AA09FDE,COMPONENT,architecture,COMPONENT,SWC-HCP5-PWR-001,VehiclePowerStateManager,vehiclepowerstatemanager,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-F8504D0ECACF284A,COMPONENT,architecture,COMPONENT,SWC-HCP5-WAKE-002,WakeSleepCoordinator,wakesleepcoordinator,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-1C399A982DF152F5,COMPONENT,architecture,COMPONENT,SWC-HCP5-STATE-003,VehicleStateDistributor,vehiclestatedistributor,MATCHED,DIRECT_ID,1.0,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml


In [0]:
ecu_aliases_df = (
    canonical_ecus_df
    .filter(F.col("ecu_name").isNotNull())
    .select(
        "canonical_id",

        F.lit("ECU").alias("entity_type"),

        F.col("ecu_name").alias(
            "alias_value"
        ),

        normalized_value("ecu_name").alias(
            "normalized_alias"
        ),

        F.lit("architecture").alias(
            "source_family"
        ),

        F.lit("AUTHORITATIVE_NAME").alias(
            "alias_type"
        ),

        F.lit("MATCHED").alias(
            "mapping_status"
        ),

        F.lit("DIRECT_ID").alias(
            "mapping_method"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        "_generation_id",
        "_source_file",
    )
)

component_aliases_df = (
    canonical_components_df
    .filter(F.col("component_name").isNotNull())
    .select(
        "canonical_id",

        F.lit("COMPONENT").alias(
            "entity_type"
        ),

        F.col("component_name").alias(
            "alias_value"
        ),

        normalized_value("component_name").alias(
            "normalized_alias"
        ),

        F.lit("architecture").alias(
            "source_family"
        ),

        F.lit("AUTHORITATIVE_NAME").alias(
            "alias_type"
        ),

        F.lit("MATCHED").alias(
            "mapping_status"
        ),

        F.lit("DIRECT_ID").alias(
            "mapping_method"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        "_generation_id",
        "_source_file",
    )
)

aliases_df = (
    ecu_aliases_df
    .unionByName(component_aliases_df)
)

display(aliases_df)

canonical_id,entity_type,alias_value,normalized_alias,source_family,alias_type,mapping_status,mapping_method,mapping_version,_generation_id,_source_file
CAN-ECU-B23A779303D7B8A0,ECU,HCP3,hcp3,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-ECU-24DD72AFD1D44768,ECU,HCP5,hcp5,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-ECU-6714038EEAC88885,ECU,ConMod,conmod,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-DB1BD195E0FECE45,COMPONENT,BluetoothConnectionManager,bluetoothconnectionmanager,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-B1CF366DEC7F8B20,COMPONENT,ConnectivityStatusService,connectivitystatusservice,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-9F79C1A5AEC97E29,COMPONENT,MediaSessionManager,mediasessionmanager,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-EAD09678DB459E65,COMPONENT,UserProfileManager,userprofilemanager,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-A4C4A48E5AA09FDE,COMPONENT,VehiclePowerStateManager,vehiclepowerstatemanager,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-F8504D0ECACF284A,COMPONENT,WakeSleepCoordinator,wakesleepcoordinator,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml
CAN-CMP-1C399A982DF152F5,COMPONENT,VehicleStateDistributor,vehiclestatedistributor,architecture,AUTHORITATIVE_NAME,MATCHED,DIRECT_ID,v1,8573dcf32f2ed486,dbfs:/Volumes/ontologysim/landing/source_files/medium/8573dcf32f2ed486/architecture/architecture.arxml


In [0]:
write_generation(
    canonical_ecus_df,
    "ontologysim.canonical.ecus",
)

write_generation(
    canonical_components_df,
    "ontologysim.canonical.components",
)

write_generation(
    identity_map_df,
    "ontologysim.canonical.identity_map",
)

write_generation(
    aliases_df,
    "ontologysim.canonical.aliases",
)

print("Canonical Architecture tables written")

Canonical Architecture tables written


In [0]:
canonical_ecus = (
    spark.table("ontologysim.canonical.ecus")
    .filter(F.col("_generation_id") == generation_id)
)

canonical_components = (
    spark.table("ontologysim.canonical.components")
    .filter(F.col("_generation_id") == generation_id)
)

identity_map = (
    spark.table("ontologysim.canonical.identity_map")
    .filter(F.col("_generation_id") == generation_id)
)

aliases = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
)


silver_ecu_rows = silver_ecus_df.count()
canonical_ecu_rows = canonical_ecus.count()

silver_component_rows = silver_components_df.count()
canonical_component_rows = canonical_components.count()


duplicate_ecu_ids = (
    canonical_ecus
    .groupBy("canonical_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_component_ids = (
    canonical_components
    .groupBy("canonical_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


ecu_identity_rows = (
    identity_map
    .filter(
        F.col("canonical_entity_type") == "ECU"
    )
    .count()
)

component_identity_rows = (
    identity_map
    .filter(
        F.col("canonical_entity_type")
        == "COMPONENT"
    )
    .count()
)


unresolved_component_ecus = (
    canonical_components
    .filter(
        F.col("source_ecu_id").isNotNull()
        & F.col("canonical_ecu_id").isNull()
    )
    .count()
)


missing_mapping_versions = (
    identity_map
    .filter(F.col("mapping_version").isNull())
    .count()
)


print(
    f"ECUs:             "
    f"{silver_ecu_rows} -> {canonical_ecu_rows}"
)

print(
    f"Components:       "
    f"{silver_component_rows} -> "
    f"{canonical_component_rows}"
)

print(
    f"ECU mappings:     {ecu_identity_rows}"
)

print(
    f"Component mappings: "
    f"{component_identity_rows}"
)

print(
    f"Aliases:          {aliases.count()}"
)

print(
    f"Duplicate ECU canonical IDs: "
    f"{duplicate_ecu_ids}"
)

print(
    f"Duplicate Component canonical IDs: "
    f"{duplicate_component_ids}"
)

print(
    f"Unresolved Component -> ECU refs: "
    f"{unresolved_component_ecus}"
)

print(
    f"Missing mapping versions: "
    f"{missing_mapping_versions}"
)


if silver_ecu_rows != canonical_ecu_rows:
    raise ValueError("ECU canonical row mismatch")

if silver_component_rows != canonical_component_rows:
    raise ValueError(
        "Component canonical row mismatch"
    )

if duplicate_ecu_ids:
    raise ValueError(
        "Duplicate canonical ECU IDs"
    )

if duplicate_component_ids:
    raise ValueError(
        "Duplicate canonical Component IDs"
    )

if ecu_identity_rows != canonical_ecu_rows:
    raise ValueError(
        "ECU identity-map coverage failed"
    )

if component_identity_rows != canonical_component_rows:
    raise ValueError(
        "Component identity-map coverage failed"
    )

if missing_mapping_versions:
    raise ValueError(
        "Missing mapping versions"
    )

print(
    "Canonical Architecture validation passed"
)

ECUs:             3 -> 3
Components:       1600 -> 1600
ECU mappings:     3
Component mappings: 1600
Aliases:          1603
Duplicate ECU canonical IDs: 0
Duplicate Component canonical IDs: 0
Unresolved Component -> ECU refs: 0
Missing mapping versions: 0
Canonical Architecture validation passed


In [0]:
silver_requirements_df = (
    spark.table("ontologysim.silver.requirements")
    .filter(F.col("_generation_id") == generation_id)
)

silver_test_cases_df = (
    spark.table("ontologysim.silver.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

silver_test_executions_df = (
    spark.table("ontologysim.silver.test_executions")
    .filter(F.col("_generation_id") == generation_id)
)

silver_defects_df = (
    spark.table("ontologysim.silver.defects")
    .filter(F.col("_generation_id") == generation_id)
)

print("Requirements:", silver_requirements_df.count())
print("TestCases:", silver_test_cases_df.count())
print("TestExecutions:", silver_test_executions_df.count())
print("Defects:", silver_defects_df.count())

Requirements: 8040
TestCases: 16080
TestExecutions: 160800
Defects: 3216


In [0]:
def canonical_id_for_source(
    prefix,
    entity_type,
    source_family,
    source_id_column,
):
    digest = F.sha2(
        F.concat_ws(
            "||",
            F.lit(entity_type),
            F.lit(source_family),
            F.col(source_id_column),
        ),
        256,
    )

    return F.concat(
        F.lit(prefix),
        F.upper(F.substring(digest, 1, 16)),
    )


def aggregate_existing_string(
    df,
    candidates,
    output_name,
):
    for candidate in candidates:
        if candidate in df.columns:
            return F.max(
                F.trim(
                    F.col(candidate).cast("string")
                )
            ).alias(output_name)

    return F.max(
        F.lit(None).cast("string")
    ).alias(output_name)


def build_authoritative_canonical(
    df,
    source_id_column,
    source_id_alias,
    entity_type,
    source_family,
    prefix,
    fields,
):
    valid_df = df.filter(
        F.col(source_id_column).isNotNull()
        & (F.trim(F.col(source_id_column)) != "")
    )

    aggregations = [
        aggregate_existing_string(
            df,
            candidates,
            output_name,
        )
        for output_name, candidates in fields.items()
    ]

    aggregations.extend(
        [
            F.max("_source_file").alias("_source_file"),
            F.max("_generation_id").alias(
                "_generation_id"
            ),
            F.max("_profile").alias("_profile"),
        ]
    )

    grouped_df = (
        valid_df
        .groupBy(
            F.trim(
                F.col(source_id_column)
            ).alias(source_id_alias)
        )
        .agg(*aggregations)
    )

    return (
        grouped_df
        .withColumn(
            "canonical_id",
            canonical_id_for_source(
                prefix,
                entity_type,
                source_family,
                source_id_alias,
            ),
        )
        .withColumn(
            "authoritative_source",
            F.lit(source_family.upper()),
        )
        .withColumn(
            "mapping_version",
            F.lit(mapping_version),
        )
        .withColumn(
            "_canonical_transformed_at",
            F.current_timestamp(),
        )
    )

In [0]:
canonical_requirements_df = (
    build_authoritative_canonical(
        df=silver_requirements_df,
        source_id_column="requirement_id",
        source_id_alias="source_requirement_id",
        entity_type="REQUIREMENT",
        source_family="requirements",
        prefix="CAN-REQ-",
        fields={
            "display_name": ["title", "requirement_title"],
            "requirement_type": ["requirement_type"],
            "priority": ["priority"],
            "status": ["status"],
            "target_ecu": ["target_ecu"],
            "target_component": ["target_component"],
        },
    )
)

write_generation(
    canonical_requirements_df,
    "ontologysim.canonical.requirements",
)

In [0]:
canonical_test_cases_df = (
    build_authoritative_canonical(
        df=silver_test_cases_df,
        source_id_column="test_case_id",
        source_id_alias="source_test_case_id",
        entity_type="TEST_CASE",
        source_family="test_management",
        prefix="CAN-TC-",
        fields={
            "display_name": [
                "title",
                "name",
                "test_case_name",
            ],
            "test_type": ["test_type"],
            "priority": ["priority"],
            "status": ["status"],
            "target_ecu": ["target_ecu"],
            "target_component": ["target_component"],
            "component_refs": ["component_refs"],
        },
    )
)

write_generation(
    canonical_test_cases_df,
    "ontologysim.canonical.test_cases",
)

In [0]:
canonical_test_executions_df = (
    build_authoritative_canonical(
        df=silver_test_executions_df,
        source_id_column="execution_id",
        source_id_alias="source_execution_id",
        entity_type="TEST_EXECUTION",
        source_family="test_execution",
        prefix="CAN-EXEC-",
        fields={
            "display_name": [
                "title",
                "name",
            ],
            "source_test_case_id": [
                "test_case_id",
            ],
            "result": [
                "result",
                "status",
            ],
            "vehicle_id": [
                "vehicle_id",
            ],
        },
    )
)

write_generation(
    canonical_test_executions_df,
    "ontologysim.canonical.test_executions",
)

In [0]:
canonical_defects_df = (
    build_authoritative_canonical(
        df=silver_defects_df,
        source_id_column="defect_id",
        source_id_alias="source_defect_id",
        entity_type="DEFECT",
        source_family="defects",
        prefix="CAN-DEF-",
        fields={
            "display_name": ["title"],
            "severity": ["severity"],
            "priority": ["priority"],
            "status": ["status"],
            "affected_ecu": ["affected_ecu"],
            "affected_component": [
                "affected_component"
            ],
            "duplicate_of": ["duplicate_of"],
        },
    )
)

write_generation(
    canonical_defects_df,
    "ontologysim.canonical.defects",
)

In [0]:
def build_identity_mapping(
    canonical_df,
    entity_type,
    source_family,
    source_id_column,
):
    return (
        canonical_df
        .select(
            "canonical_id",

            F.lit(entity_type).alias(
                "canonical_entity_type"
            ),

            F.lit(source_family).alias(
                "source_family"
            ),

            F.lit(entity_type).alias(
                "source_entity_type"
            ),

            F.col(source_id_column).alias(
                "source_id"
            ),

            F.coalesce(
                F.col("display_name"),
                F.col(source_id_column),
            ).alias("source_name"),

            F.lower(
                F.regexp_replace(
                    F.coalesce(
                        F.col("display_name"),
                        F.col(source_id_column),
                    ),
                    r"[^A-Za-z0-9]+",
                    "",
                )
            ).alias("normalized_source_value"),

            F.lit("MATCHED").alias(
                "mapping_status"
            ),

            F.lit("DIRECT_ID").alias(
                "mapping_method"
            ),

            F.lit(1.0).alias(
                "mapping_confidence"
            ),

            F.lit(mapping_version).alias(
                "mapping_version"
            ),

            "_generation_id",
            "_source_file",
        )
    )


new_identity_maps = [
    build_identity_mapping(
        canonical_requirements_df,
        "REQUIREMENT",
        "requirements",
        "source_requirement_id",
    ),
    build_identity_mapping(
        canonical_test_cases_df,
        "TEST_CASE",
        "test_management",
        "source_test_case_id",
    ),
    build_identity_mapping(
        canonical_test_executions_df,
        "TEST_EXECUTION",
        "test_execution",
        "source_execution_id",
    ),
    build_identity_mapping(
        canonical_defects_df,
        "DEFECT",
        "defects",
        "source_defect_id",
    ),
]

existing_identity_map_df = (
    spark.table("ontologysim.canonical.identity_map")
    .filter(F.col("_generation_id") == generation_id)
    .filter(
        ~F.col("canonical_entity_type").isin(
            "REQUIREMENT",
            "TEST_CASE",
            "TEST_EXECUTION",
            "DEFECT",
        )
    )
)

combined_identity_map_df = existing_identity_map_df

for new_df in new_identity_maps:
    combined_identity_map_df = (
        combined_identity_map_df
        .unionByName(new_df)
    )

write_generation(
    combined_identity_map_df,
    "ontologysim.canonical.identity_map",
)

print("Identity map updated")

Identity map updated


In [0]:
def validate_canonical_entity(
    silver_df,
    source_id_column,
    canonical_table,
    entity_type,
):
    source_rows = silver_df.count()

    null_ids = (
        silver_df
        .filter(
            F.col(source_id_column).isNull()
            | (
                F.trim(
                    F.col(source_id_column)
                ) == ""
            )
        )
        .count()
    )

    distinct_source_ids = (
        silver_df
        .filter(
            F.col(source_id_column).isNotNull()
            & (
                F.trim(
                    F.col(source_id_column)
                ) != ""
            )
        )
        .select(
            F.trim(
                F.col(source_id_column)
            ).alias("source_id")
        )
        .distinct()
        .count()
    )

    canonical_df = (
        spark.table(canonical_table)
        .filter(
            F.col("_generation_id")
            == generation_id
        )
    )

    canonical_rows = canonical_df.count()

    duplicate_canonical_ids = (
        canonical_df
        .groupBy("canonical_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    identity_rows = (
        spark.table(
            "ontologysim.canonical.identity_map"
        )
        .filter(
            (F.col("_generation_id") == generation_id)
            & (
                F.col("canonical_entity_type")
                == entity_type
            )
        )
        .count()
    )

    duplicates_collapsed = (
        source_rows - distinct_source_ids - null_ids
    )

    print(f"\n{entity_type}")
    print(f"  Silver rows:          {source_rows}")
    print(
        f"  Distinct source IDs:  "
        f"{distinct_source_ids}"
    )
    print(
        f"  Duplicate rows:       "
        f"{duplicates_collapsed}"
    )
    print(f"  Missing IDs:          {null_ids}")
    print(
        f"  Canonical entities:   "
        f"{canonical_rows}"
    )
    print(
        f"  Identity mappings:    "
        f"{identity_rows}"
    )
    print(
        f"  Duplicate canonical IDs: "
        f"{duplicate_canonical_ids}"
    )

    if null_ids:
        raise ValueError(
            f"{entity_type}: missing authoritative IDs"
        )

    if canonical_rows != distinct_source_ids:
        raise ValueError(
            f"{entity_type}: canonical count mismatch"
        )

    if identity_rows != canonical_rows:
        raise ValueError(
            f"{entity_type}: identity-map mismatch"
        )

    if duplicate_canonical_ids:
        raise ValueError(
            f"{entity_type}: duplicate canonical IDs"
        )


validate_canonical_entity(
    silver_requirements_df,
    "requirement_id",
    "ontologysim.canonical.requirements",
    "REQUIREMENT",
)

validate_canonical_entity(
    silver_test_cases_df,
    "test_case_id",
    "ontologysim.canonical.test_cases",
    "TEST_CASE",
)

validate_canonical_entity(
    silver_test_executions_df,
    "execution_id",
    "ontologysim.canonical.test_executions",
    "TEST_EXECUTION",
)

validate_canonical_entity(
    silver_defects_df,
    "defect_id",
    "ontologysim.canonical.defects",
    "DEFECT",
)

identity_total = (
    spark.table(
        "ontologysim.canonical.identity_map"
    )
    .filter(
        F.col("_generation_id") == generation_id
    )
    .count()
)

print(
    f"\nTotal identity mappings: {identity_total}"
)

print(
    "Canonical authoritative entities validation passed"
)


REQUIREMENT
  Silver rows:          8040
  Distinct source IDs:  8000
  Duplicate rows:       40
  Missing IDs:          0
  Canonical entities:   8000
  Identity mappings:    8000
  Duplicate canonical IDs: 0

TEST_CASE
  Silver rows:          16080
  Distinct source IDs:  16000
  Duplicate rows:       80
  Missing IDs:          0
  Canonical entities:   16000
  Identity mappings:    16000
  Duplicate canonical IDs: 0

TEST_EXECUTION
  Silver rows:          160800
  Distinct source IDs:  160000
  Duplicate rows:       800
  Missing IDs:          0
  Canonical entities:   160000
  Identity mappings:    160000
  Duplicate canonical IDs: 0

DEFECT
  Silver rows:          3216
  Distinct source IDs:  3200
  Duplicate rows:       16
  Missing IDs:          0
  Canonical entities:   3200
  Identity mappings:    3200
  Duplicate canonical IDs: 0

Total identity mappings: 188803
Canonical authoritative entities validation passed


In [0]:
def normalize_expr(column):
    return F.lower(
        F.regexp_replace(
            F.trim(column.cast("string")),
            r"[^A-Za-z0-9]+",
            "",
        )
    )


identity_current_df = (
    spark.table("ontologysim.canonical.identity_map")
    .filter(F.col("_generation_id") == generation_id)
)

authoritative_aliases_df = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
    .filter(F.col("alias_type") == "AUTHORITATIVE_NAME")
)


direct_lookup_df = (
    identity_current_df
    .select(
        F.col("canonical_entity_type").alias("entity_type"),
        normalize_expr(F.col("source_id")).alias("lookup_key"),
        "canonical_id",
        F.lit(1).alias("priority"),
    )
)

name_lookup_df = (
    authoritative_aliases_df
    .select(
        "entity_type",
        F.col("normalized_alias").alias("lookup_key"),
        "canonical_id",
        F.lit(2).alias("priority"),
    )
)

lookup_candidates_df = (
    direct_lookup_df
    .unionByName(name_lookup_df)
    .filter(F.col("lookup_key") != "")
)

resolution_lookup_df = (
    lookup_candidates_df
    .groupBy("entity_type", "lookup_key")
    .agg(
        F.collect_set("canonical_id").alias("_canonical_ids"),
        F.min("priority").alias("_priority"),
    )
    .withColumn(
        "candidate_count",
        F.size("_canonical_ids"),
    )
    .withColumn(
        "resolved_canonical_id",
        F.when(
            F.col("candidate_count") == 1,
            F.element_at("_canonical_ids", 1),
        ),
    )
    .withColumn(
        "mapping_method",
        F.when(
            (F.col("candidate_count") == 1)
            & (F.col("_priority") == 1),
            F.lit("DIRECT_ID"),
        ).when(
            F.col("candidate_count") == 1,
            F.lit("NORMALIZED_NAME"),
        ),
    )
    .drop("_canonical_ids", "_priority")
)

print("Resolution lookup ready")

Resolution lookup ready


In [0]:
def alias_candidates(
    df,
    column_name,
    entity_type,
    source_family,
):
    if column_name not in df.columns:
        return None

    return (
        df
        .select(
            F.lit(entity_type).alias("entity_type"),
            F.lit(source_family).alias("source_family"),
            F.trim(
                F.col(column_name).cast("string")
            ).alias("alias_value"),
            "_source_file",
        )
        .filter(
            F.col("alias_value").isNotNull()
            & (F.col("alias_value") != "")
        )
        .dropDuplicates(
            [
                "entity_type",
                "source_family",
                "alias_value",
            ]
        )
    )


candidate_sets = []

candidate_specs = [
    (
        silver_requirements_df,
        "target_ecu",
        "ECU",
        "requirements",
    ),
    (
        silver_requirements_df,
        "target_component",
        "COMPONENT",
        "requirements",
    ),
    (
        silver_test_cases_df,
        "target_ecu",
        "ECU",
        "test_management",
    ),
    (
        silver_test_cases_df,
        "component_refs",
        "COMPONENT",
        "test_management",
    ),
    (
        silver_defects_df,
        "affected_ecu",
        "ECU",
        "defects",
    ),
    (
        silver_defects_df,
        "affected_component",
        "COMPONENT",
        "defects",
    ),
    (
        spark.table("ontologysim.silver.hcp3_runtime")
        .filter(F.col("_generation_id") == generation_id),
        "ecu_id",
        "ECU",
        "hcp3_runtime",
    ),
    (
        spark.table("ontologysim.silver.hcp3_runtime")
        .filter(F.col("_generation_id") == generation_id),
        "component_name",
        "COMPONENT",
        "hcp3_runtime",
    ),
    (
        spark.table("ontologysim.silver.hcp5_runtime")
        .filter(F.col("_generation_id") == generation_id),
        "ecu",
        "ECU",
        "hcp5_runtime",
    ),
    (
        spark.table("ontologysim.silver.hcp5_runtime")
        .filter(F.col("_generation_id") == generation_id),
        "component",
        "COMPONENT",
        "hcp5_runtime",
    ),
    (
        spark.table("ontologysim.silver.conmod_runtime")
        .filter(F.col("_generation_id") == generation_id),
        "ecu",
        "ECU",
        "conmod_runtime",
    ),
]

for spec in candidate_specs:
    candidate_df = alias_candidates(*spec)

    if candidate_df is not None:
        candidate_sets.append(candidate_df)


all_candidate_aliases_df = candidate_sets[0]

for candidate_df in candidate_sets[1:]:
    all_candidate_aliases_df = (
        all_candidate_aliases_df
        .unionByName(candidate_df)
    )

all_candidate_aliases_df = (
    all_candidate_aliases_df
    .dropDuplicates(
        [
            "entity_type",
            "source_family",
            "alias_value",
        ]
    )
    .withColumn(
        "normalized_alias",
        normalize_expr(F.col("alias_value")),
    )
)

print(
    "Distinct source aliases:",
    all_candidate_aliases_df.count(),
)

Distinct source aliases: 2014


In [0]:
resolved_alias_candidates_df = (
    all_candidate_aliases_df.alias("a")
    .join(
        resolution_lookup_df.alias("r"),
        (
            F.col("a.entity_type")
            == F.col("r.entity_type")
        )
        & (
            F.col("a.normalized_alias")
            == F.col("r.lookup_key")
        ),
        "left",
    )
    .select(
        F.col("r.resolved_canonical_id").alias(
            "canonical_id"
        ),
        F.col("a.entity_type"),
        F.col("a.alias_value"),
        F.col("a.normalized_alias"),
        F.col("a.source_family"),

        F.lit("SOURCE_REFERENCE").alias(
            "alias_type"
        ),

        F.when(
            F.col("r.candidate_count").isNull(),
            F.lit("UNRESOLVED"),
        ).when(
            F.col("r.candidate_count") > 1,
            F.lit("AMBIGUOUS"),
        ).otherwise(
            F.lit("MATCHED")
        ).alias("mapping_status"),

        F.col("r.mapping_method"),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        F.lit(generation_id).alias(
            "_generation_id"
        ),

        F.col("a._source_file"),
    )
)


combined_aliases_df = (
    authoritative_aliases_df
    .unionByName(
        resolved_alias_candidates_df
    )
    .dropDuplicates(
        [
            "entity_type",
            "source_family",
            "alias_value",
            "mapping_version",
        ]
    )
)


write_generation(
    combined_aliases_df,
    "ontologysim.canonical.aliases",
)

display(
    combined_aliases_df
    .groupBy(
        "entity_type",
        "mapping_status",
    )
    .count()
    .orderBy(
        "entity_type",
        "mapping_status",
    )
)

entity_type,mapping_status,count
COMPONENT,MATCHED,3555
COMPONENT,UNRESOLVED,47
ECU,MATCHED,15


In [0]:
current_aliases_df = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
)


matched_alias_lookup_df = (
    current_aliases_df
    .filter(
        (F.col("mapping_status") == "MATCHED")
        & F.col("canonical_id").isNotNull()
    )
    .select(
        "entity_type",
        F.col("normalized_alias").alias(
            "lookup_key"
        ),
        "canonical_id",
        F.when(
            F.col("mapping_method") == "DIRECT_ID",
            F.lit(1),
        ).otherwise(
            F.lit(2)
        ).alias("priority"),
    )
)


relationship_lookup_candidates_df = (
    direct_lookup_df
    .unionByName(matched_alias_lookup_df)
)


relationship_lookup_df = (
    relationship_lookup_candidates_df
    .groupBy(
        "entity_type",
        "lookup_key",
    )
    .agg(
        F.collect_set("canonical_id").alias(
            "_canonical_ids"
        ),
        F.min("priority").alias("_priority"),
    )
    .withColumn(
        "candidate_count",
        F.size("_canonical_ids"),
    )
    .withColumn(
        "resolved_canonical_id",
        F.when(
            F.col("candidate_count") == 1,
            F.element_at("_canonical_ids", 1),
        ),
    )
    .withColumn(
        "mapping_method",
        F.when(
            (F.col("candidate_count") == 1)
            & (F.col("_priority") == 1),
            F.lit("DIRECT_ID"),
        ).when(
            F.col("candidate_count") == 1,
            F.lit("NORMALIZED_NAME"),
        ),
    )
    .drop("_canonical_ids", "_priority")
)

print("Relationship lookup ready")

Relationship lookup ready


In [0]:
def build_relationships(
    source_df,
    source_id_column,
    source_entity_type,
    target_column,
    target_entity_type,
    relationship_type,
    source_family,
    multi_value=False,
):
    if (
        source_id_column not in source_df.columns
        or target_column not in source_df.columns
    ):
        return None

    base_df = (
        source_df
        .select(
            F.trim(
                F.col(source_id_column).cast("string")
            ).alias("source_record_id"),

            F.col(target_column)
            .cast("string")
            .alias("_target_raw"),

            "_source_file",
        )
        .filter(
            F.col("source_record_id").isNotNull()
            & (F.col("source_record_id") != "")
            & F.col("_target_raw").isNotNull()
            & (F.trim("_target_raw") != "")
        )
    )

    if multi_value:
        base_df = (
            base_df
            .withColumn(
                "_target_clean",
                F.regexp_replace(
                    F.col("_target_raw"),
                    r"""[\[\]"']""",
                    "",
                ),
            )
            .withColumn(
                "target_source_reference",
                F.explode(
                    F.split(
                        F.col("_target_clean"),
                        r"\s*[,;|]\s*",
                    )
                ),
            )
        )
    else:
        base_df = base_df.withColumn(
            "target_source_reference",
            F.trim(F.col("_target_raw")),
        )

    base_df = (
        base_df
        .filter(
            F.col(
                "target_source_reference"
            ).isNotNull()
            & (
                F.trim(
                    F.col(
                        "target_source_reference"
                    )
                ) != ""
            )
        )
        .dropDuplicates(
            [
                "source_record_id",
                "target_source_reference",
            ]
        )
        .withColumn(
            "_source_lookup_key",
            normalize_expr(
                F.col("source_record_id")
            ),
        )
        .withColumn(
            "_target_lookup_key",
            normalize_expr(
                F.col("target_source_reference")
            ),
        )
    )


    source_lookup = (
        identity_current_df
        .filter(
            F.col("canonical_entity_type")
            == source_entity_type
        )
        .select(
            normalize_expr(
                F.col("source_id")
            ).alias("_source_lookup_key"),
            F.col("canonical_id").alias(
                "source_canonical_id"
            ),
        )
        .dropDuplicates(
            ["_source_lookup_key"]
        )
    )


    target_lookup = (
        relationship_lookup_df
        .filter(
            F.col("entity_type")
            == target_entity_type
        )
        .select(
            F.col("lookup_key").alias(
                "_target_lookup_key"
            ),
            F.col("resolved_canonical_id").alias(
                "target_canonical_id"
            ),
            "candidate_count",
            "mapping_method",
        )
    )


    result_df = (
        base_df
        .join(
            source_lookup,
            "_source_lookup_key",
            "left",
        )
        .join(
            target_lookup,
            "_target_lookup_key",
            "left",
        )
        .withColumn(
            "source_entity_type",
            F.lit(source_entity_type),
        )
        .withColumn(
            "target_entity_type",
            F.lit(target_entity_type),
        )
        .withColumn(
            "relationship_type",
            F.lit(relationship_type),
        )
        .withColumn(
            "relationship_origin",
            F.lit("EXPLICIT"),
        )
        .withColumn(
            "relationship_status",
            F.when(
                F.col("source_canonical_id").isNull(),
                F.lit("BROKEN"),
            ).when(
                F.col("candidate_count").isNull(),
                F.lit("UNRESOLVED"),
            ).when(
                F.col("candidate_count") > 1,
                F.lit("AMBIGUOUS"),
            ).otherwise(
                F.lit("RESOLVED")
            ),
        )
        .withColumn(
            "mapping_confidence",
            F.when(
                F.col("mapping_method")
                == "DIRECT_ID",
                F.lit(1.0),
            ).when(
                F.col("mapping_method")
                == "NORMALIZED_NAME",
                F.lit(0.95),
            ),
        )
        .withColumn(
            "source_family",
            F.lit(source_family),
        )
        .withColumn(
            "mapping_version",
            F.lit(mapping_version),
        )
        .withColumn(
            "_generation_id",
            F.lit(generation_id),
        )
        .withColumn(
            "relationship_id",
            F.concat(
                F.lit("CAN-REL-"),
                F.upper(
                    F.substring(
                        F.sha2(
                            F.concat_ws(
                                "||",
                                F.lit(
                                    source_entity_type
                                ),
                                F.col(
                                    "source_record_id"
                                ),
                                F.lit(
                                    relationship_type
                                ),
                                F.lit(
                                    target_entity_type
                                ),
                                F.col(
                                    "target_source_reference"
                                ),
                                F.lit(source_family),
                                F.lit(mapping_version),
                            ),
                            256,
                        ),
                        1,
                        16,
                    )
                ),
            ),
        )
        .select(
            "relationship_id",
            "source_canonical_id",
            "source_entity_type",
            "relationship_type",
            "target_canonical_id",
            "target_entity_type",
            "target_source_reference",
            "relationship_origin",
            "relationship_status",
            "mapping_method",
            "mapping_confidence",
            "source_family",
            "source_record_id",
            "mapping_version",
            "_generation_id",
            "_source_file",
        )
    )

    return result_df

In [0]:
relationship_sets = []


relationship_specs = [
    # Architecture
    (
        silver_components_df,
        "ecu_id",
        "ECU",
        "component_id",
        "COMPONENT",
        "HOSTS",
        "architecture",
        False,
    ),

    # Requirements
    (
        silver_requirements_df,
        "requirement_id",
        "REQUIREMENT",
        "target_ecu",
        "ECU",
        "TARGETS",
        "requirements",
        False,
    ),
    (
        silver_requirements_df,
        "requirement_id",
        "REQUIREMENT",
        "target_component",
        "COMPONENT",
        "TARGETS",
        "requirements",
        False,
    ),
    (
        silver_requirements_df,
        "requirement_id",
        "REQUIREMENT",
        "parent_requirement_id",
        "REQUIREMENT",
        "DERIVES_FROM",
        "requirements",
        False,
    ),
    (
        silver_requirements_df,
        "requirement_id",
        "REQUIREMENT",
        "related_requirement_ids",
        "REQUIREMENT",
        "RELATES_TO",
        "requirements",
        True,
    ),

    # Test cases
    (
        silver_test_cases_df,
        "test_case_id",
        "TEST_CASE",
        "target_ecu",
        "ECU",
        "TARGETS",
        "test_management",
        False,
    ),
    (
        silver_test_cases_df,
        "test_case_id",
        "TEST_CASE",
        "component_refs",
        "COMPONENT",
        "TARGETS",
        "test_management",
        True,
    ),

    # Executions
    (
        silver_test_executions_df,
        "execution_id",
        "TEST_EXECUTION",
        "test_case_id",
        "TEST_CASE",
        "EXECUTION_OF",
        "test_execution",
        False,
    ),

    # Defects
    (
        silver_defects_df,
        "defect_id",
        "DEFECT",
        "affected_ecu",
        "ECU",
        "AFFECTS",
        "defects",
        False,
    ),
    (
        silver_defects_df,
        "defect_id",
        "DEFECT",
        "affected_component",
        "COMPONENT",
        "AFFECTS",
        "defects",
        False,
    ),
    (
        silver_defects_df,
        "defect_id",
        "DEFECT",
        "duplicate_of",
        "DEFECT",
        "DUPLICATE_OF",
        "defects",
        False,
    ),
    (
        silver_defects_df,
        "defect_id",
        "DEFECT",
        "requirement_refs",
        "REQUIREMENT",
        "RELATES_TO",
        "defects",
        True,
    ),
    (
        silver_defects_df,
        "defect_id",
        "DEFECT",
        "test_case_refs",
        "TEST_CASE",
        "RELATES_TO",
        "defects",
        True,
    ),
    (
        silver_defects_df,
        "defect_id",
        "DEFECT",
        "test_execution_refs",
        "TEST_EXECUTION",
        "FOUND_DURING",
        "defects",
        True,
    ),
]


for spec in relationship_specs:
    rel_df = build_relationships(*spec)

    if rel_df is not None:
        relationship_sets.append(rel_df)


relationships_df = relationship_sets[0]

for rel_df in relationship_sets[1:]:
    relationships_df = (
        relationships_df
        .unionByName(rel_df)
    )


relationships_df = (
    relationships_df
    .dropDuplicates(["relationship_id"])
)


write_generation(
    relationships_df,
    "ontologysim.canonical.relationships",
)

print(
    "Canonical relationships:",
    relationships_df.count(),
)

Canonical relationships: 219907


In [0]:
aliases_current = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
)

relationships_current = (
    spark.table("ontologysim.canonical.relationships")
    .filter(F.col("_generation_id") == generation_id)
)


print("ALIAS RESOLUTION")
display(
    aliases_current
    .groupBy(
        "entity_type",
        "mapping_status",
    )
    .count()
    .orderBy(
        "entity_type",
        "mapping_status",
    )
)


print("RELATIONSHIP RESOLUTION")
display(
    relationships_current
    .groupBy(
        "relationship_type",
        "relationship_status",
    )
    .count()
    .orderBy(
        "relationship_type",
        "relationship_status",
    )
)


duplicate_relationship_ids = (
    relationships_current
    .groupBy("relationship_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

broken_relationships = (
    relationships_current
    .filter(
        F.col("relationship_status") == "BROKEN"
    )
    .count()
)

invalid_resolved = (
    relationships_current
    .filter(
        (F.col("relationship_status") == "RESOLVED")
        & (
            F.col("source_canonical_id").isNull()
            | F.col("target_canonical_id").isNull()
        )
    )
    .count()
)

print(
    "Duplicate relationship IDs:",
    duplicate_relationship_ids,
)

print(
    "Broken relationships:",
    broken_relationships,
)

print(
    "Invalid RESOLVED relationships:",
    invalid_resolved,
)


if duplicate_relationship_ids:
    raise ValueError(
        "Duplicate canonical relationship IDs"
    )

if broken_relationships:
    raise ValueError(
        "Canonical source identities are missing"
    )

if invalid_resolved:
    raise ValueError(
        "Resolved relationship has missing canonical ID"
    )

print(
    "Canonical relationship validation passed"
)

ALIAS RESOLUTION


entity_type,mapping_status,count
COMPONENT,MATCHED,3555
COMPONENT,UNRESOLVED,47
ECU,MATCHED,15


RELATIONSHIP RESOLUTION


relationship_type,relationship_status,count
AFFECTS,RESOLVED,5735
AFFECTS,UNRESOLVED,665
EXECUTION_OF,RESOLVED,155200
FOUND_DURING,RESOLVED,2947
HOSTS,RESOLVED,1600
RELATES_TO,RESOLVED,6400
TARGETS,RESOLVED,39116
TARGETS,UNRESOLVED,8244


Duplicate relationship IDs: 0
Broken relationships: 0
Invalid RESOLVED relationships: 0
Canonical relationship validation passed


In [0]:
print("UNRESOLVED COMPONENT ALIASES")

display(
    aliases_current
    .filter(
        (F.col("entity_type") == "COMPONENT")
        & (F.col("mapping_status") == "UNRESOLVED")
    )
    .select(
        "source_family",
        "alias_value",
        "normalized_alias",
    )
    .distinct()
    .orderBy(
        "source_family",
        "alias_value",
    )
)


print("UNRESOLVED RELATIONSHIP REFERENCES")

display(
    relationships_current
    .filter(
        F.col("relationship_status") == "UNRESOLVED"
    )
    .groupBy(
        "relationship_type",
        "target_entity_type",
        "target_source_reference",
    )
    .count()
    .orderBy(
        F.col("count").desc(),
    )
)

UNRESOLVED COMPONENT ALIASES


source_family,alias_value,normalized_alias
defects,BT reconnect module,btreconnectmodule
defects,BT reconnect module legacy,btreconnectmodulelegacy
defects,CellularManager legacy,cellularmanagerlegacy
defects,ChargingStateManager legacy,chargingstatemanagerlegacy
defects,ConnectivityStatusService legacy,connectivitystatusservicelegacy
defects,InfotainmentStartupCoordinator legacy,infotainmentstartupcoordinatorlegacy
defects,NavigationPositionService legacy,navigationpositionservicelegacy
defects,SoftwareCompatibilityChecker legacy,softwarecompatibilitycheckerlegacy
defects,WakeSleepCoordinator legacy,wakesleepcoordinatorlegacy
hcp3_runtime,BT_CONN_MGR,btconnmgr


UNRESOLVED RELATIONSHIP REFERENCES


relationship_type,target_entity_type,target_source_reference,count
TARGETS,COMPONENT,BackendService,2174
TARGETS,COMPONENT,NetworkDiagnostics,1535
TARGETS,COMPONENT,VehicleStateService,1068
TARGETS,COMPONENT,EnergyService,1040
TARGETS,COMPONENT,ConnectivityStatus,1000
TARGETS,COMPONENT,WakeCoordinator,934
AFFECTS,COMPONENT,BT reconnect module,633
TARGETS,COMPONENT,MediaManager,405
AFFECTS,COMPONENT,BT reconnect module legacy,7
TARGETS,COMPONENT,ConnectivityStatusService_Legacy,7


In [0]:
# ─────────────────────────────────────────────────
# V2 SEMANTIC GAP DIAGNOSTICS
# Pre-alias-rule inspection (rules NOT yet applied for V2)
# ─────────────────────────────────────────────────

aliases_v2 = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
)

relationships_v2 = (
    spark.table("ontologysim.canonical.relationships")
    .filter(F.col("_generation_id") == generation_id)
)

canonical_components_v2 = (
    spark.table("ontologysim.canonical.components")
    .filter(F.col("_generation_id") == generation_id)
)

canonical_test_cases_v2 = (
    spark.table("ontologysim.canonical.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

# ═══════════════════════════════════════════════════
# 1. UNRESOLVED / AMBIGUOUS COMPONENT ALIASES
#    With alias value, normalized alias, source family,
#    occurrence count in source Silver tables
# ═══════════════════════════════════════════════════

unresolved_aliases = (
    aliases_v2
    .filter(
        (F.col("entity_type") == "COMPONENT")
        & (F.col("mapping_status").isin("UNRESOLVED", "AMBIGUOUS"))
    )
)

# Count occurrences of each unresolved alias in Silver source tables
silver_component_alias_sources = []
for spec in candidate_specs:
    silver_df, col_name, ent_type, src_family = spec
    if ent_type == "COMPONENT" and col_name in silver_df.columns:
        src_counts = (
            silver_df
            .select(F.trim(F.col(col_name).cast("string")).alias("alias_value"))
            .filter(F.col("alias_value").isNotNull() & (F.col("alias_value") != ""))
            .groupBy("alias_value")
            .count()
            .withColumnRenamed("count", "occurrence_count")
            .withColumn("source_family", F.lit(src_family))
        )
        silver_component_alias_sources.append(src_counts)

if silver_component_alias_sources:
    alias_occurrences = silver_component_alias_sources[0]
    for s in silver_component_alias_sources[1:]:
        alias_occurrences = alias_occurrences.unionByName(s)
    alias_occurrences = (
        alias_occurrences
        .groupBy("alias_value", "source_family")
        .agg(F.sum("occurrence_count").alias("occurrence_count"))
    )
else:
    alias_occurrences = spark.createDataFrame([], "alias_value string, source_family string, occurrence_count long")

print("═" * 60)
print("1. UNRESOLVED / AMBIGUOUS COMPONENT ALIASES")
print("═" * 60)

display(
    unresolved_aliases
    .join(
        alias_occurrences,
        ["alias_value", "source_family"],
        "left",
    )
    .select(
        "alias_value",
        "normalized_alias",
        "source_family",
        "alias_type",
        "mapping_status",
        F.coalesce(F.col("occurrence_count"), F.lit(0)).alias("occurrence_count"),
    )
    .orderBy("source_family", "alias_value")
)

# ═══════════════════════════════════════════════════
# 2. UNRESOLVED RELATIONSHIPS
#    Grouped by relationship_type, source_entity_type,
#    target_entity_type, target_source_reference, count
# ═══════════════════════════════════════════════════

print("═" * 60)
print("2. UNRESOLVED RELATIONSHIPS (grouped)")
print("═" * 60)

display(
    relationships_v2
    .filter(F.col("relationship_status").isin("UNRESOLVED", "AMBIGUOUS"))
    .groupBy(
        "relationship_type",
        "source_entity_type",
        "target_entity_type",
        "target_source_reference",
    )
    .count()
    .orderBy(F.col("count").desc())
)

# ═══════════════════════════════════════════════════
# 3. TESTCASE → COMPONENT COVERAGE
# ═══════════════════════════════════════════════════

total_test_cases = canonical_test_cases_v2.count()

tc_component_rels = (
    relationships_v2
    .filter(
        (F.col("source_entity_type") == "TEST_CASE")
        & (F.col("relationship_type") == "TARGETS")
        & (F.col("target_entity_type") == "COMPONENT")
    )
)

tc_resolved = tc_component_rels.filter(F.col("relationship_status") == "RESOLVED").count()
tc_unresolved = tc_component_rels.filter(F.col("relationship_status") == "UNRESOLVED").count()
tc_ambiguous = tc_component_rels.filter(F.col("relationship_status") == "AMBIGUOUS").count()

tc_distinct_unresolved_aliases = (
    tc_component_rels
    .filter(F.col("relationship_status") == "UNRESOLVED")
    .select("target_source_reference")
    .distinct()
    .count()
)

tc_testcases_with_component = (
    tc_component_rels
    .filter(F.col("relationship_status") == "RESOLVED")
    .select("source_record_id")
    .distinct()
    .count()
)

print("═" * 60)
print("3. TESTCASE → COMPONENT COVERAGE")
print("═" * 60)
print(f"Total canonical TestCases:              {total_test_cases}")
print(f"Resolved TARGETS → Component rels:      {tc_resolved}")
print(f"Unresolved TARGETS → Component rels:      {tc_unresolved}")
print(f"Ambiguous TARGETS → Component rels:       {tc_ambiguous}")
print(f"Distinct unresolved component aliases:   {tc_distinct_unresolved_aliases}")
print(f"TestCases with ≥1 resolved Component:     {tc_testcases_with_component}")

print()
print("Unresolved TestCase → Component references:")
display(
    tc_component_rels
    .filter(F.col("relationship_status") == "UNRESOLVED")
    .groupBy("target_source_reference")
    .count()
    .orderBy(F.col("count").desc())
)

# ═══════════════════════════════════════════════════
# 4. COMPARE UNRESOLVED ALIASES AGAINST
#    AUTHORITATIVE ARCHITECTURE COMPONENTS
#    Using normalized component name, source component ID,
#    and ECU context
# ═══════════════════════════════════════════════════

arch_components = (
    canonical_components_v2
    .select(
        "canonical_id",
        "source_component_id",
        "component_name",
        "source_ecu_id",
        "canonical_ecu_id",
        normalize_expr(F.col("component_name")).alias("norm_component_name"),
        normalize_expr(F.col("source_component_id")).alias("norm_source_id"),
    )
)

unresolved_with_context = (
    unresolved_aliases
    .select("alias_value", "normalized_alias", "source_family")
    .distinct()
)

# Try matching: normalized_alias == norm_component_name
name_matches = (
    unresolved_with_context.alias("u")
    .join(
        arch_components.alias("c"),
        F.col("u.normalized_alias") == F.col("c.norm_component_name"),
        "left",
    )
    .select(
        F.col("u.alias_value"),
        F.col("u.normalized_alias"),
        F.col("u.source_family"),
        F.col("c.component_name").alias("arch_component_name"),
        F.col("c.source_component_id").alias("arch_source_id"),
        F.col("c.source_ecu_id").alias("arch_ecu"),
        F.col("c.canonical_id").alias("matched_canonical_id"),
        F.lit("NAME_MATCH").alias("match_type"),
    )
)

# Try matching: normalized_alias == norm_source_id
id_matches = (
    unresolved_with_context.alias("u")
    .join(
        arch_components.alias("c"),
        F.col("u.normalized_alias") == F.col("c.norm_source_id"),
        "left",
    )
    .select(
        F.col("u.alias_value"),
        F.col("u.normalized_alias"),
        F.col("u.source_family"),
        F.col("c.component_name").alias("arch_component_name"),
        F.col("c.source_component_id").alias("arch_source_id"),
        F.col("c.source_ecu_id").alias("arch_ecu"),
        F.col("c.canonical_id").alias("matched_canonical_id"),
        F.lit("ID_MATCH").alias("match_type"),
    )
)

# Combine matches, prioritizing name matches
all_matches = name_matches.unionByName(id_matches)

# For each unresolved alias, check if any match found
resolved_via_arch = (
    all_matches
    .filter(F.col("matched_canonical_id").isNotNull())
    .dropDuplicates(["alias_value", "normalized_alias", "source_family"])
)

still_unresolved = (
    unresolved_with_context
    .join(
        resolved_via_arch.select("alias_value", "normalized_alias", "source_family"),
        ["alias_value", "normalized_alias", "source_family"],
        "left_anti",
    )
)

print("═" * 60)
print("4. COMPARISON AGAINST ARCHITECTURE COMPONENTS")
print("═" * 60)

matched_count = resolved_via_arch.count()
unmatched_count = still_unresolved.count()

print(f"Unresolved aliases with architecture match:  {matched_count}")
print(f"Unresolved aliases with NO architecture match: {unmatched_count}")
print()
print("── Aliases MATCHED against architecture components ──")
display(
    resolved_via_arch
    .select(
        "alias_value",
        "normalized_alias",
        "source_family",
        "arch_component_name",
        "arch_source_id",
        "arch_ecu",
        "matched_canonical_id",
        "match_type",
    )
    .orderBy("source_family", "alias_value")
)

print("── Aliases with NO architecture match (genuinely ambiguous) ──")
display(
    still_unresolved
    .orderBy("source_family", "alias_value")
)

════════════════════════════════════════════════════════════
1. UNRESOLVED / AMBIGUOUS COMPONENT ALIASES
════════════════════════════════════════════════════════════


alias_value,normalized_alias,source_family,alias_type,mapping_status,occurrence_count
BT reconnect module,btreconnectmodule,defects,SOURCE_REFERENCE,UNRESOLVED,635
BT reconnect module legacy,btreconnectmodulelegacy,defects,SOURCE_REFERENCE,UNRESOLVED,7
CellularManager legacy,cellularmanagerlegacy,defects,SOURCE_REFERENCE,UNRESOLVED,4
ChargingStateManager legacy,chargingstatemanagerlegacy,defects,SOURCE_REFERENCE,UNRESOLVED,3
ConnectivityStatusService legacy,connectivitystatusservicelegacy,defects,SOURCE_REFERENCE,UNRESOLVED,7
InfotainmentStartupCoordinator legacy,infotainmentstartupcoordinatorlegacy,defects,SOURCE_REFERENCE,UNRESOLVED,3
NavigationPositionService legacy,navigationpositionservicelegacy,defects,SOURCE_REFERENCE,UNRESOLVED,5
SoftwareCompatibilityChecker legacy,softwarecompatibilitycheckerlegacy,defects,SOURCE_REFERENCE,UNRESOLVED,2
WakeSleepCoordinator legacy,wakesleepcoordinatorlegacy,defects,SOURCE_REFERENCE,UNRESOLVED,1
BT_CONN_MGR,btconnmgr,hcp3_runtime,SOURCE_REFERENCE,UNRESOLVED,96331


════════════════════════════════════════════════════════════
2. UNRESOLVED RELATIONSHIPS (grouped)
════════════════════════════════════════════════════════════


relationship_type,source_entity_type,target_entity_type,target_source_reference,count
TARGETS,TEST_CASE,COMPONENT,BackendService,2174
TARGETS,TEST_CASE,COMPONENT,NetworkDiagnostics,1535
TARGETS,TEST_CASE,COMPONENT,VehicleStateService,1068
TARGETS,TEST_CASE,COMPONENT,EnergyService,1040
TARGETS,TEST_CASE,COMPONENT,ConnectivityStatus,1000
TARGETS,TEST_CASE,COMPONENT,WakeCoordinator,934
AFFECTS,DEFECT,COMPONENT,BT reconnect module,633
TARGETS,TEST_CASE,COMPONENT,MediaManager,405
AFFECTS,DEFECT,COMPONENT,BT reconnect module legacy,7
TARGETS,REQUIREMENT,COMPONENT,ConnectivityStatusService_Legacy,7


════════════════════════════════════════════════════════════
3. TESTCASE → COMPONENT COVERAGE
════════════════════════════════════════════════════════════
Total canonical TestCases:              16000
Resolved TARGETS → Component rels:      7830
Unresolved TARGETS → Component rels:      8170
Ambiguous TARGETS → Component rels:       0
Distinct unresolved component aliases:   10
TestCases with ≥1 resolved Component:     7830

Unresolved TestCase → Component references:


target_source_reference,count
BackendService,2174
NetworkDiagnostics,1535
VehicleStateService,1068
EnergyService,1040
ConnectivityStatus,1000
WakeCoordinator,934
MediaManager,405
ProfileManager,6
PowerStateManager,5
BluetoothManager,3


════════════════════════════════════════════════════════════
4. COMPARISON AGAINST ARCHITECTURE COMPONENTS
════════════════════════════════════════════════════════════
Unresolved aliases with architecture match:  0
Unresolved aliases with NO architecture match: 47

── Aliases MATCHED against architecture components ──


alias_value,normalized_alias,source_family,arch_component_name,arch_source_id,arch_ecu,matched_canonical_id,match_type


── Aliases with NO architecture match (genuinely ambiguous) ──


alias_value,normalized_alias,source_family
BT reconnect module,btreconnectmodule,defects
BT reconnect module legacy,btreconnectmodulelegacy,defects
CellularManager legacy,cellularmanagerlegacy,defects
ChargingStateManager legacy,chargingstatemanagerlegacy,defects
ConnectivityStatusService legacy,connectivitystatusservicelegacy,defects
InfotainmentStartupCoordinator legacy,infotainmentstartupcoordinatorlegacy,defects
NavigationPositionService legacy,navigationpositionservicelegacy,defects
SoftwareCompatibilityChecker legacy,softwarecompatibilitycheckerlegacy,defects
WakeSleepCoordinator legacy,wakesleepcoordinatorlegacy,defects
BT_CONN_MGR,btconnmgr,hcp3_runtime


In [0]:
alias_rules = [
    (
        "defects",
        "BT reconnect module",
        "BluetoothConnectionManager",
    ),
    (
        "defects",
        "BT reconnect module legacy",
        "BluetoothConnectionManager",
    ),
    (
        "hcp3_runtime",
        "BT_CONN_MGR",
        "BluetoothConnectionManager",
    ),
    (
        "hcp3_runtime",
        "CONN_STATUS_SVC",
        "ConnectivityStatusService",
    ),
    (
        "requirements",
        "AuxiliaryComponent0002_Legacy",
        "AuxiliaryComponent0002",
    ),
    (
        "requirements",
        "AuxiliaryComponent0045_Legacy",
        "AuxiliaryComponent0045",
    ),
    (
        "requirements",
        "ConnectivityStatusService_Legacy",
        "ConnectivityStatusService",
    ),
    (
        "requirements",
        "MediaSessionManager_Legacy",
        "MediaSessionManager",
    ),
    # V2 source-pattern extensions
    (
        "hcp3_runtime",
        "MEDIA_SESS_MGR",
        "MediaSessionManager",
    ),
    (
        "defects",
        "ConnectivityStatusService legacy",
        "ConnectivityStatusService",
    ),
    (
        "requirements",
        "AuxiliaryComponent0050_Legacy",
        "AuxiliaryComponent0050",
    ),
    (
        "requirements",
        "AuxiliaryComponent0081_Legacy",
        "AuxiliaryComponent0081",
    ),
    (
        "requirements",
        "InfotainmentStartupCoordinator_Legacy",
        "InfotainmentStartupCoordinator",
    ),
    # MEDIUM-specific extensions — same legacy/_Legacy pattern
    # as DEV V2 rules, covering aliases unique to MEDIUM data
    (
        "defects",
        "CellularManager legacy",
        "CellularManager",
    ),
    (
        "defects",
        "ChargingStateManager legacy",
        "ChargingStateManager",
    ),
    (
        "defects",
        "InfotainmentStartupCoordinator legacy",
        "InfotainmentStartupCoordinator",
    ),
    (
        "defects",
        "NavigationPositionService legacy",
        "NavigationPositionService",
    ),
    (
        "defects",
        "SoftwareCompatibilityChecker legacy",
        "SoftwareCompatibilityChecker",
    ),
    (
        "defects",
        "WakeSleepCoordinator legacy",
        "WakeSleepCoordinator",
    ),
    (
        "requirements",
        "AuxiliaryComponent0353_Legacy",
        "AuxiliaryComponent0353",
    ),
    (
        "requirements",
        "AuxiliaryComponent0428_Legacy",
        "AuxiliaryComponent0428",
    ),
    (
        "requirements",
        "AuxiliaryComponent0476_Legacy",
        "AuxiliaryComponent0476",
    ),
    (
        "requirements",
        "AuxiliaryComponent0670_Legacy",
        "AuxiliaryComponent0670",
    ),
    (
        "requirements",
        "AuxiliaryComponent0675_Legacy",
        "AuxiliaryComponent0675",
    ),
    (
        "requirements",
        "AuxiliaryComponent0731_Legacy",
        "AuxiliaryComponent0731",
    ),
    (
        "requirements",
        "AuxiliaryComponent0857_Legacy",
        "AuxiliaryComponent0857",
    ),
    (
        "requirements",
        "AuxiliaryComponent0873_Legacy",
        "AuxiliaryComponent0873",
    ),
    (
        "requirements",
        "AuxiliaryComponent1062_Legacy",
        "AuxiliaryComponent1062",
    ),
    (
        "requirements",
        "AuxiliaryComponent1134_Legacy",
        "AuxiliaryComponent1134",
    ),
    (
        "requirements",
        "AuxiliaryComponent1386_Legacy",
        "AuxiliaryComponent1386",
    ),
    (
        "requirements",
        "AuxiliaryComponent1429_Legacy",
        "AuxiliaryComponent1429",
    ),
    (
        "requirements",
        "BackendCommunicationService_Legacy",
        "BackendCommunicationService",
    ),
    (
        "requirements",
        "CellularManager_Legacy",
        "CellularManager",
    ),
    (
        "requirements",
        "ChargingStateManager_Legacy",
        "ChargingStateManager",
    ),
    (
        "requirements",
        "ConnectivityService_Legacy",
        "ConnectivityService",
    ),
    (
        "requirements",
        "EnergyManagementService_Legacy",
        "EnergyManagementService",
    ),
    (
        "requirements",
        "NavigationPositionService_Legacy",
        "NavigationPositionService",
    ),
    (
        "requirements",
        "NetworkDiagnosticsService_Legacy",
        "NetworkDiagnosticsService",
    ),
    (
        "requirements",
        "SoftwareCompatibilityChecker_Legacy",
        "SoftwareCompatibilityChecker",
    ),
    (
        "requirements",
        "VehicleStateDistributor_Legacy",
        "VehicleStateDistributor",
    ),
    (
        "requirements",
        "WakeSleepCoordinator_Legacy",
        "WakeSleepCoordinator",
    ),
]


alias_rules_df = spark.createDataFrame(
    alias_rules,
    [
        "source_family",
        "alias_value",
        "canonical_component_name",
    ],
).withColumn(
    "normalized_alias",
    normalize_expr(F.col("alias_value")),
).withColumn(
    "_canonical_name_key",
    normalize_expr(
        F.col("canonical_component_name")
    ),
)


component_name_lookup_df = (
    spark.table("ontologysim.canonical.components")
    .filter(F.col("_generation_id") == generation_id)
    .select(
        "canonical_id",
        "component_name",
        normalize_expr(
            F.col("component_name")
        ).alias("_canonical_name_key"),
    )
)


simple_resolved_df = (
    alias_rules_df.alias("rule")
    .join(
        component_name_lookup_df.alias("component"),
        "_canonical_name_key",
        "left",
    )
    .select(
        F.col("component.canonical_id").alias(
            "canonical_id"
        ),

        F.lit("COMPONENT").alias(
            "entity_type"
        ),

        F.col("rule.alias_value"),

        F.col("rule.normalized_alias"),

        F.col("rule.source_family"),

        F.lit("SOURCE_ALIAS").alias(
            "alias_type"
        ),

        F.when(
            F.col("component.canonical_id").isNotNull(),
            F.lit("MATCHED"),
        ).otherwise(
            F.lit("UNRESOLVED")
        ).alias("mapping_status"),

        F.when(
            F.col("component.canonical_id").isNotNull(),
            F.lit("ALIAS_RULE"),
        ).alias("mapping_method"),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        F.lit(generation_id).alias(
            "_generation_id"
        ),

        F.lit(None).cast("string").alias(
            "_source_file"
        ),
    )
)


# ── V2 ECU-context rules (test_management) ──
# Resolved by matching both alias value AND ECU context
# from the test case's target_ecu field.
ecu_context_rules = [
    (
        "test_management",
        "BackendService",
        "BackendCommunicationService",
        "CONMOD",
    ),
    (
        "test_management",
        "BluetoothManager",
        "BluetoothConnectionManager",
        "HCP3",
    ),
    (
        "test_management",
        "ConnectivityStatus",
        "ConnectivityStatusService",
        "HCP3",
    ),
    (
        "test_management",
        "EnergyService",
        "EnergyManagementService",
        "HCP5",
    ),
    (
        "test_management",
        "MediaManager",
        "MediaSessionManager",
        "HCP3",
    ),
    (
        "test_management",
        "NetworkDiagnostics",
        "NetworkDiagnosticsService",
        "CONMOD",
    ),
    (
        "test_management",
        "PowerStateManager",
        "VehiclePowerStateManager",
        "HCP5",
    ),
    (
        "test_management",
        "ProfileManager",
        "UserProfileManager",
        "HCP3",
    ),
    (
        "test_management",
        "VehicleStateService",
        "VehicleStateDistributor",
        "HCP5",
    ),
    (
        "test_management",
        "WakeCoordinator",
        "WakeSleepCoordinator",
        "HCP5",
    ),
]


ecu_rules_df = (
    spark.createDataFrame(
        ecu_context_rules,
        [
            "source_family",
            "alias_value",
            "canonical_component_name",
            "expected_ecu_context",
        ],
    )
    .withColumn(
        "normalized_alias",
        normalize_expr(F.col("alias_value")),
    )
    .withColumn(
        "_canonical_name_key",
        normalize_expr(
            F.col("canonical_component_name")
        ),
    )
)


# Get actual ECU context for each test_management alias
# from silver test_cases target_ecu field
tc_ecu_context_df = (
    spark.table("ontologysim.silver.test_cases")
    .filter(F.col("_generation_id") == generation_id)
    .select(
        F.trim(
            F.col("component_refs").cast("string")
        ).alias("alias_value"),
        F.trim(
            F.col("target_ecu").cast("string")
        ).alias("actual_ecu_context"),
    )
    .filter(
        F.col("alias_value").isNotNull()
        & (F.col("alias_value") != "")
    )
    .distinct()
)


ecu_resolved_df = (
    ecu_rules_df.alias("rule")
    .join(
        tc_ecu_context_df.alias("tc"),
        F.col("rule.alias_value")
        == F.col("tc.alias_value"),
        "left",
    )
    .join(
        component_name_lookup_df.alias("component"),
        F.col("rule._canonical_name_key")
        == F.col("component._canonical_name_key"),
        "left",
    )
    .select(
        F.col("component.canonical_id").alias(
            "canonical_id"
        ),
        F.lit("COMPONENT").alias(
            "entity_type"
        ),
        F.col("rule.alias_value"),
        F.col("rule.normalized_alias"),
        F.col("rule.source_family"),
        F.lit("SOURCE_ALIAS").alias(
            "alias_type"
        ),
        F.when(
            F.col("component.canonical_id").isNotNull()
            & (
                F.col("tc.actual_ecu_context")
                == F.col("rule.expected_ecu_context")
            ),
            F.lit("MATCHED"),
        ).otherwise(
            F.lit("UNRESOLVED")
        ).alias("mapping_status"),
        F.when(
            F.col("component.canonical_id").isNotNull()
            & (
                F.col("tc.actual_ecu_context")
                == F.col("rule.expected_ecu_context")
            ),
            F.lit("ALIAS_RULE_ECU_CONTEXT"),
        ).alias("mapping_method"),
        F.lit(mapping_version).alias(
            "mapping_version"
        ),
        F.lit(generation_id).alias(
            "_generation_id"
        ),
        F.lit(None).cast("string").alias(
            "_source_file"
        ),
    )
)


# Combine simple + ECU-context rules
resolved_rules_df = simple_resolved_df.unionByName(
    ecu_resolved_df
)


display(
    resolved_rules_df.select(
        "source_family",
        "alias_value",
        "canonical_id",
        "mapping_status",
        "mapping_method",
    ).orderBy(
        "source_family",
        "alias_value",
    )
)


unresolved_rules = (
    resolved_rules_df.filter(
        F.col("mapping_status") == "UNRESOLVED"
    ).count()
)

print(
    "Unresolved alias rules:",
    unresolved_rules,
)

if unresolved_rules:
    raise ValueError(
        "Alias rules could not be resolved — "
        "check target components and ECU context"
    )

print(
    "All alias rules resolved successfully"
)

source_family,alias_value,canonical_id,mapping_status,mapping_method
defects,BT reconnect module,CAN-CMP-DB1BD195E0FECE45,MATCHED,ALIAS_RULE
defects,BT reconnect module legacy,CAN-CMP-DB1BD195E0FECE45,MATCHED,ALIAS_RULE
defects,CellularManager legacy,CAN-CMP-99C1D69D5AD519AB,MATCHED,ALIAS_RULE
defects,ChargingStateManager legacy,CAN-CMP-B332A335BEE7F81B,MATCHED,ALIAS_RULE
defects,ConnectivityStatusService legacy,CAN-CMP-B1CF366DEC7F8B20,MATCHED,ALIAS_RULE
defects,InfotainmentStartupCoordinator legacy,CAN-CMP-0D1764BDAF5B9E8E,MATCHED,ALIAS_RULE
defects,NavigationPositionService legacy,CAN-CMP-D5DF1F428924853E,MATCHED,ALIAS_RULE
defects,SoftwareCompatibilityChecker legacy,CAN-CMP-D4D060653BFE2337,MATCHED,ALIAS_RULE
defects,WakeSleepCoordinator legacy,CAN-CMP-F8504D0ECACF284A,MATCHED,ALIAS_RULE
hcp3_runtime,BT_CONN_MGR,CAN-CMP-DB1BD195E0FECE45,MATCHED,ALIAS_RULE


Unresolved alias rules: 0
All alias rules resolved successfully


In [0]:
existing_aliases_df = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
)


rule_keys_df = (
    resolved_rules_df
    .select(
        "entity_type",
        "source_family",
        "normalized_alias",
    )
    .distinct()
)


aliases_without_replaced_rows_df = (
    existing_aliases_df.alias("a")
    .join(
        rule_keys_df.alias("r"),
        (
            F.col("a.entity_type")
            == F.col("r.entity_type")
        )
        & (
            F.col("a.source_family")
            == F.col("r.source_family")
        )
        & (
            F.col("a.normalized_alias")
            == F.col("r.normalized_alias")
        ),
        "left_anti",
    )
)


updated_aliases_df = (
    aliases_without_replaced_rows_df
    .unionByName(resolved_rules_df)
)


write_generation(
    updated_aliases_df,
    "ontologysim.canonical.aliases",
)

print("Alias rules persisted")

Alias rules persisted


In [0]:
aliases_current = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
)


display(
    aliases_current
    .groupBy(
        "entity_type",
        "mapping_status",
    )
    .count()
    .orderBy(
        "entity_type",
        "mapping_status",
    )
)


unresolved_components = (
    aliases_current
    .filter(
        (F.col("entity_type") == "COMPONENT")
        & (
            F.col("mapping_status")
            == "UNRESOLVED"
        )
    )
    .count()
)

print(
    "Unresolved Component aliases:",
    unresolved_components,
)

if unresolved_components:
    raise ValueError(
        "Component aliases remain unresolved"
    )

print("Deterministic alias validation passed")

entity_type,mapping_status,count
COMPONENT,MATCHED,3606
ECU,MATCHED,15


Unresolved Component aliases: 0
Deterministic alias validation passed


In [0]:
from pyspark.sql import Window
import uuid


def existing_col(df, candidates, alias, data_type="string"):
    for candidate in candidates:
        if candidate in df.columns:
            return (
                F.col(candidate)
                .cast(data_type)
                .alias(alias)
            )

    return F.lit(None).cast(data_type).alias(alias)


def standardize_runtime(df, source_family, column_map):
    # Build fingerprint only from source/Silver business fields.
    # Ingestion timestamps/run IDs are deliberately excluded.
    business_columns = sorted(
        column
        for column in df.columns
        if not column.startswith("_")
    )

    fingerprint_struct = F.struct(
        *[
            F.coalesce(
                F.col(column).cast("string"),
                F.lit("<NULL>"),
            ).alias(column)
            for column in business_columns
        ]
    )

    prepared = (
        df
        .withColumn(
            "_source_signature",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.lit(source_family),
                    F.to_json(fingerprint_struct),
                ),
                256,
            ),
        )
    )

    duplicate_window = (
        Window
        .partitionBy("_source_signature")
        .orderBy(F.lit(1))
    )

    prepared = (
        prepared
        .withColumn(
            "_duplicate_ordinal",
            F.row_number().over(duplicate_window),
        )
        .withColumn(
            "source_record_id",
            F.concat_ws(
                ":",
                F.lit(source_family),
                F.col("_source_signature"),
                F.lpad(
                    F.col("_duplicate_ordinal").cast("string"),
                    6,
                    "0",
                ),
            ),
        )
    )

    return (
        prepared
        .select(
            F.concat(
                F.lit("CAN-EVT-"),
                F.upper(
                    F.substring(
                        F.sha2("source_record_id", 256),
                        1,
                        20,
                    )
                ),
            ).alias("canonical_id"),

            F.col("source_record_id"),

            F.lit(source_family).alias(
                "runtime_source"
            ),

            existing_col(
                prepared,
                column_map["timestamp"],
                "event_timestamp",
                "timestamp",
            ),

            existing_col(
                prepared,
                column_map["ecu"],
                "source_ecu",
            ),

            existing_col(
                prepared,
                column_map["component"],
                "source_component",
            ),

            existing_col(
                prepared,
                column_map["event"],
                "event_type",
            ),

            existing_col(
                prepared,
                column_map["severity"],
                "severity",
            ),

            existing_col(
                prepared,
                column_map["message"],
                "message",
            ),

            existing_col(
                prepared,
                column_map["session"],
                "session_id",
            ),

            existing_col(
                prepared,
                column_map["execution"],
                "execution_id",
            ),

            existing_col(
                prepared,
                column_map["software"],
                "software_version",
            ),

            existing_col(
                prepared,
                column_map.get("network_type", []),
                "network_type",
            ),

            existing_col(
                prepared,
                column_map.get("connection_state", []),
                "connection_state",
            ),

            existing_col(
                prepared,
                column_map.get("backend_state", []),
                "backend_state",
            ),

            existing_col(
                prepared,
                column_map.get("service", []),
                "service",
            ),

            existing_col(
                prepared,
                column_map.get("signal_strength", []),
                "signal_strength",
                "long",
            ),

            "_source_file",
            "_generation_id",
            "_profile",
        )
        .withColumn(
            "mapping_version",
            F.lit(mapping_version),
        )
        .withColumn(
            "_canonical_transformed_at",
            F.current_timestamp(),
        )
    )

In [0]:
hcp3_df = (
    spark.table("ontologysim.silver.hcp3_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

hcp5_df = (
    spark.table("ontologysim.silver.hcp5_runtime")
    .filter(F.col("_generation_id") == generation_id)
)

conmod_df = (
    spark.table("ontologysim.silver.conmod_runtime")
    .filter(F.col("_generation_id") == generation_id)
)


hcp3_runtime_df = standardize_runtime(
    hcp3_df,
    "hcp3_runtime",
    {
        "timestamp": ["timestamp"],
        "ecu": ["ecu_id", "ecu"],
        "component": ["component_name", "component"],
        "event": ["event_type", "event"],
        "severity": ["severity"],
        "message": ["message"],
        "session": ["session_id", "session"],
        "execution": ["execution_id", "execution"],
        "software": ["software_version", "sw"],
    },
)


hcp5_runtime_df = standardize_runtime(
    hcp5_df,
    "hcp5_runtime",
    {
        "timestamp": ["timestamp"],
        "ecu": ["ecu", "ecu_id"],
        "component": ["component", "component_name"],
        "event": ["event", "event_type"],
        "severity": ["severity"],
        "message": ["message"],
        "session": ["session", "session_id"],
        "execution": ["execution", "execution_id"],
        "software": ["software_version", "sw"],
    },
)


conmod_runtime_df = standardize_runtime(
    conmod_df,
    "conmod_runtime",
    {
        "timestamp": ["timestamp"],
        "ecu": ["ecu", "ecu_id"],
        "component": ["component", "component_name"],
        "event": ["event", "event_type"],
        "severity": ["severity"],
        "message": ["message"],
        "session": ["session", "session_id"],
        "execution": ["execution", "execution_id"],
        "software": ["software_version", "software", "sw"],
        "network_type": ["network_type"],
        "connection_state": ["connection_state"],
        "backend_state": ["backend_state"],
        "service": ["service"],
        "signal_strength": ["signal_strength"],
    },
)


runtime_events_base_df = (
    hcp3_runtime_df
    .unionByName(hcp5_runtime_df)
    .unionByName(conmod_runtime_df)
)

print(
    "Standardized RuntimeEvents:",
    runtime_events_base_df.count(),
)

Standardized RuntimeEvents: 38548484


In [0]:
current_identity_df = (
    spark.table("ontologysim.canonical.identity_map")
    .filter(F.col("_generation_id") == generation_id)
)

current_aliases_df = (
    spark.table("ontologysim.canonical.aliases")
    .filter(F.col("_generation_id") == generation_id)
    .filter(F.col("mapping_status") == "MATCHED")
)


direct_reference_df = (
    current_identity_df
    .filter(
        F.col("canonical_entity_type").isin(
            "ECU",
            "COMPONENT",
        )
    )
    .select(
        F.col("canonical_entity_type").alias(
            "entity_type"
        ),
        normalize_expr(
            F.col("source_id")
        ).alias("lookup_key"),
        "canonical_id",
    )
)


alias_reference_df = (
    current_aliases_df
    .filter(
        F.col("entity_type").isin(
            "ECU",
            "COMPONENT",
        )
    )
    .select(
        "entity_type",
        F.col("normalized_alias").alias(
            "lookup_key"
        ),
        "canonical_id",
    )
)


runtime_reference_lookup_df = (
    direct_reference_df
    .unionByName(alias_reference_df)
    .groupBy(
        "entity_type",
        "lookup_key",
    )
    .agg(
        F.collect_set("canonical_id").alias(
            "_ids"
        )
    )
    .withColumn(
        "candidate_count",
        F.size("_ids"),
    )
    .withColumn(
        "resolved_id",
        F.when(
            F.col("candidate_count") == 1,
            F.element_at("_ids", 1),
        ),
    )
    .drop("_ids")
)


ecu_runtime_lookup_df = (
    runtime_reference_lookup_df
    .filter(F.col("entity_type") == "ECU")
    .select(
        F.col("lookup_key").alias(
            "_ecu_lookup_key"
        ),
        F.col("resolved_id").alias(
            "canonical_ecu_id"
        ),
        F.col("candidate_count").alias(
            "_ecu_candidates"
        ),
    )
)


component_runtime_lookup_df = (
    runtime_reference_lookup_df
    .filter(F.col("entity_type") == "COMPONENT")
    .select(
        F.col("lookup_key").alias(
            "_component_lookup_key"
        ),
        F.col("resolved_id").alias(
            "canonical_component_id"
        ),
        F.col("candidate_count").alias(
            "_component_candidates"
        ),
    )
)


runtime_events_resolved_df = (
    runtime_events_base_df
    .withColumn(
        "_ecu_lookup_key",
        normalize_expr(F.col("source_ecu")),
    )
    .withColumn(
        "_component_lookup_key",
        normalize_expr(F.col("source_component")),
    )
    .join(
        ecu_runtime_lookup_df,
        "_ecu_lookup_key",
        "left",
    )
    .join(
        component_runtime_lookup_df,
        "_component_lookup_key",
        "left",
    )
    .withColumn(
        "ecu_mapping_status",
        F.when(
            F.col("source_ecu").isNull(),
            F.lit("NOT_APPLICABLE"),
        ).when(
            F.col("canonical_ecu_id").isNotNull(),
            F.lit("MATCHED"),
        ).otherwise(
            F.lit("UNRESOLVED"),
        ),
    )
    .withColumn(
        "component_mapping_status",
        F.when(
            F.col("source_component").isNull(),
            F.lit("NOT_APPLICABLE"),
        ).when(
            F.col("canonical_component_id").isNotNull(),
            F.lit("MATCHED"),
        ).otherwise(
            F.lit("UNRESOLVED"),
        ),
    )
    .drop(
        "_ecu_lookup_key",
        "_component_lookup_key",
        "_ecu_candidates",
        "_component_candidates",
    )
)

In [0]:
runtime_execution_summary_df = (
    runtime_events_resolved_df
    .filter(
        F.col("execution_id").isNotNull()
        & (F.trim("execution_id") != "")
    )
    .groupBy(
        F.trim("execution_id").alias(
            "source_execution_id"
        )
    )
    .agg(
        F.min("event_timestamp").alias(
            "first_event_at"
        ),
        F.max("event_timestamp").alias(
            "last_event_at"
        ),
        F.count("*").alias(
            "runtime_event_count"
        ),
        F.array_sort(
            F.collect_set("runtime_source")
        ).alias("runtime_sources"),
        F.array_sort(
            F.collect_set("session_id")
        ).alias("session_ids"),
    )
)


canonical_executions_current_df = (
    spark.table(
        "ontologysim.canonical.test_executions"
    )
    .filter(F.col("_generation_id") == generation_id)
    .select(
        "canonical_id",
        "source_execution_id",
    )
)


incidents_df = (
    canonical_executions_current_df.alias("execution")
    .join(
        runtime_execution_summary_df.alias("runtime"),
        "source_execution_id",
        "inner",
    )
    .select(
        F.concat(
            F.lit("CAN-INC-"),
            F.upper(
                F.substring(
                    F.sha2(
                        F.concat_ws(
                            "||",
                            F.lit("INCIDENT"),
                            F.col(
                                "source_execution_id"
                            ),
                        ),
                        256,
                    ),
                    1,
                    16,
                )
            ),
        ).alias("canonical_id"),

        F.col("execution.canonical_id").alias(
            "canonical_test_execution_id"
        ),

        "source_execution_id",

        "first_event_at",
        "last_event_at",
        "runtime_event_count",
        "runtime_sources",
        "session_ids",

        F.lit("RULE_DERIVED").alias(
            "derivation_method"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        F.lit(generation_id).alias(
            "_generation_id"
        ),

        F.lit(profile).alias(
            "_profile"
        ),
    )
    .withColumn(
        "_canonical_transformed_at",
        F.current_timestamp(),
    )
)


print(
    "Derived Incidents:",
    incidents_df.count(),
)

Derived Incidents: 160000


In [0]:
incident_lookup_df = (
    incidents_df
    .select(
        "source_execution_id",
        F.col("canonical_id").alias(
            "canonical_incident_id"
        ),
    )
)


canonical_runtime_events_df = (
    runtime_events_resolved_df.alias("event")
    .join(
        incident_lookup_df.alias("incident"),
        F.trim(F.col("event.execution_id"))
        == F.col("incident.source_execution_id"),
        "left",
    )
    .drop(
        F.col("incident.source_execution_id")
    )
)


write_generation(
    canonical_runtime_events_df,
    "ontologysim.canonical.runtime_events",
)

write_generation(
    incidents_df,
    "ontologysim.canonical.incidents",
)

print("RuntimeEvents and Incidents written")

RuntimeEvents and Incidents written


In [0]:
incident_relationships_df = (
    incidents_df
    .select(
        F.concat(
            F.lit("CAN-REL-"),
            F.upper(
                F.substring(
                    F.sha2(
                        F.concat_ws(
                            "||",
                            F.col("canonical_id"),
                            F.lit("INVOLVES"),
                            F.col(
                                "canonical_test_execution_id"
                            ),
                        ),
                        256,
                    ),
                    1,
                    16,
                )
            ),
        ).alias("relationship_id"),

        F.col("canonical_id").alias(
            "source_canonical_id"
        ),

        F.lit("INCIDENT").alias(
            "source_entity_type"
        ),

        F.lit("INVOLVES").alias(
            "relationship_type"
        ),

        F.col(
            "canonical_test_execution_id"
        ).alias(
            "target_canonical_id"
        ),

        F.lit("TEST_EXECUTION").alias(
            "target_entity_type"
        ),

        F.col("source_execution_id").alias(
            "target_source_reference"
        ),

        F.lit("RULE_DERIVED").alias(
            "relationship_origin"
        ),

        F.lit("RESOLVED").alias(
            "relationship_status"
        ),

        F.lit("REFERENCE").alias(
            "mapping_method"
        ),

        F.lit(1.0).alias(
            "mapping_confidence"
        ),

        F.lit("derived_incident").alias(
            "source_family"
        ),

        F.col("source_execution_id").alias(
            "source_record_id"
        ),

        F.lit(mapping_version).alias(
            "mapping_version"
        ),

        F.col("_generation_id"),

        F.lit(None).cast("string").alias(
            "_source_file"
        ),
    )
)


existing_relationships_df = (
    spark.table(
        "ontologysim.canonical.relationships"
    )
    .filter(F.col("_generation_id") == generation_id)
    .filter(
        F.col("source_entity_type") != "INCIDENT"
    )
)


all_relationships_df = (
    existing_relationships_df
    .unionByName(incident_relationships_df)
    .dropDuplicates(["relationship_id"])
)


write_generation(
    all_relationships_df,
    "ontologysim.canonical.relationships",
)

print(
    "Relationships after Incident extension:",
    all_relationships_df.count(),
)

Relationships after Incident extension: 379907


In [0]:
canonical_runtime_current_df = (
    spark.table(
        "ontologysim.canonical.runtime_events"
    )
    .filter(F.col("_generation_id") == generation_id)
)

canonical_incidents_current_df = (
    spark.table(
        "ontologysim.canonical.incidents"
    )
    .filter(F.col("_generation_id") == generation_id)
)


expected_runtime_rows = (
    hcp3_df.count()
    + hcp5_df.count()
    + conmod_df.count()
)

actual_runtime_rows = (
    canonical_runtime_current_df.count()
)


duplicate_runtime_ids = (
    canonical_runtime_current_df
    .groupBy("canonical_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


duplicate_incident_ids = (
    canonical_incidents_current_df
    .groupBy("canonical_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


unresolved_runtime_ecus = (
    canonical_runtime_current_df
    .filter(
        F.col("ecu_mapping_status")
        == "UNRESOLVED"
    )
    .count()
)


unresolved_runtime_components = (
    canonical_runtime_current_df
    .filter(
        F.col("component_mapping_status")
        == "UNRESOLVED"
    )
    .count()
)


runtime_with_execution = (
    canonical_runtime_current_df
    .filter(
        F.col("execution_id").isNotNull()
        & (F.trim("execution_id") != "")
    )
)


runtime_without_incident = (
    runtime_with_execution
    .filter(
        F.col("canonical_incident_id").isNull()
    )
    .count()
)


print(
    "Expected RuntimeEvents:",
    expected_runtime_rows,
)
print(
    "Canonical RuntimeEvents:",
    actual_runtime_rows,
)
print(
    "Derived Incidents:",
    canonical_incidents_current_df.count(),
)
print(
    "Duplicate RuntimeEvent IDs:",
    duplicate_runtime_ids,
)
print(
    "Duplicate Incident IDs:",
    duplicate_incident_ids,
)
print(
    "Unresolved runtime ECUs:",
    unresolved_runtime_ecus,
)
print(
    "Unresolved runtime Components:",
    unresolved_runtime_components,
)
print(
    "Runtime events with execution but no Incident:",
    runtime_without_incident,
)


if expected_runtime_rows != actual_runtime_rows:
    raise ValueError(
        "RuntimeEvent row preservation failed"
    )

if duplicate_runtime_ids:
    raise ValueError(
        "Duplicate canonical RuntimeEvent IDs"
    )

if duplicate_incident_ids:
    raise ValueError(
        "Duplicate canonical Incident IDs"
    )

print(
    "RuntimeEvent + Incident validation passed"
)

Expected RuntimeEvents: 38548484
Canonical RuntimeEvents: 38548484
Derived Incidents: 160000
Duplicate RuntimeEvent IDs: 0
Duplicate Incident IDs: 0
Unresolved runtime ECUs: 0
Unresolved runtime Components: 0
Runtime events with execution but no Incident: 0
RuntimeEvent + Incident validation passed


In [0]:
validation_run_id = str(uuid.uuid4())

relationships_current_df = (
    spark.table(
        "ontologysim.canonical.relationships"
    )
    .filter(F.col("_generation_id") == generation_id)
)

identity_current_df = (
    spark.table(
        "ontologysim.canonical.identity_map"
    )
    .filter(F.col("_generation_id") == generation_id)
)

aliases_current_df = (
    spark.table(
        "ontologysim.canonical.aliases"
    )
    .filter(F.col("_generation_id") == generation_id)
)


checks = []


def add_check(
    check_name,
    passed,
    observed,
    expected,
    details,
):
    checks.append(
        (
            check_name,
            "PASS" if passed else "FAIL",
            str(observed),
            str(expected),
            details,
        )
    )


add_check(
    "runtime_row_preservation",
    actual_runtime_rows == expected_runtime_rows,
    actual_runtime_rows,
    expected_runtime_rows,
    "Silver valid runtime rows must be preserved",
)

add_check(
    "runtime_id_uniqueness",
    duplicate_runtime_ids == 0,
    duplicate_runtime_ids,
    0,
    "Canonical RuntimeEvent IDs must be unique",
)

add_check(
    "incident_id_uniqueness",
    duplicate_incident_ids == 0,
    duplicate_incident_ids,
    0,
    "Canonical Incident IDs must be unique",
)

add_check(
    "relationship_id_uniqueness",
    (
        relationships_current_df
        .groupBy("relationship_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
        == 0
    ),
    0,
    0,
    "Canonical relationship IDs must be unique",
)

invalid_resolved_relationships = (
    relationships_current_df
    .filter(
        (F.col("relationship_status") == "RESOLVED")
        & (
            F.col("source_canonical_id").isNull()
            | F.col("target_canonical_id").isNull()
        )
    )
    .count()
)

add_check(
    "resolved_relationship_integrity",
    invalid_resolved_relationships == 0,
    invalid_resolved_relationships,
    0,
    "Resolved relationships require both canonical IDs",
)


missing_identity_mapping_versions = (
    identity_current_df
    .filter(F.col("mapping_version").isNull())
    .count()
)

add_check(
    "identity_mapping_version",
    missing_identity_mapping_versions == 0,
    missing_identity_mapping_versions,
    0,
    "Identity mappings require mapping_version",
)


missing_relationship_versions = (
    relationships_current_df
    .filter(F.col("mapping_version").isNull())
    .count()
)

add_check(
    "relationship_mapping_version",
    missing_relationship_versions == 0,
    missing_relationship_versions,
    0,
    "Relationships require mapping_version",
)


unresolved_aliases = (
    aliases_current_df
    .filter(
        F.col("mapping_status").isin(
            "UNRESOLVED",
            "AMBIGUOUS",
        )
    )
    .count()
)

add_check(
    "semantic_alias_resolution",
    unresolved_aliases == 0,
    unresolved_aliases,
    0,
    "Current deterministic ECU/Component aliases",
)


add_check(
    "runtime_ecu_resolution",
    unresolved_runtime_ecus == 0,
    unresolved_runtime_ecus,
    0,
    "Runtime ECU references",
)


add_check(
    "runtime_component_resolution",
    unresolved_runtime_components == 0,
    unresolved_runtime_components,
    0,
    "Runtime Component references",
)


add_check(
    "runtime_incident_linkage",
    runtime_without_incident == 0,
    runtime_without_incident,
    0,
    "Runtime records with execution IDs should link to derived Incidents",
)


canonical_validation_df = (
    spark.createDataFrame(
        checks,
        [
            "check_name",
            "status",
            "observed_value",
            "expected_value",
            "details",
        ],
    )
    .withColumn(
        "generation_id",
        F.lit(generation_id),
    )
    .withColumn(
        "profile",
        F.lit(profile),
    )
    .withColumn(
        "mapping_version",
        F.lit(mapping_version),
    )
    .withColumn(
        "validation_run_id",
        F.lit(validation_run_id),
    )
    .withColumn(
        "validated_at",
        F.current_timestamp(),
    )
)


(
    canonical_validation_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "ontologysim.control.canonical_validation_results"
    )
)


display(
    canonical_validation_df
    .orderBy("check_name")
)

check_name,status,observed_value,expected_value,details,generation_id,profile,mapping_version,validation_run_id,validated_at
identity_mapping_version,PASS,0,0,Identity mappings require mapping_version,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
incident_id_uniqueness,PASS,0,0,Canonical Incident IDs must be unique,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
relationship_id_uniqueness,PASS,0,0,Canonical relationship IDs must be unique,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
relationship_mapping_version,PASS,0,0,Relationships require mapping_version,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
resolved_relationship_integrity,PASS,0,0,Resolved relationships require both canonical IDs,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
runtime_component_resolution,PASS,0,0,Runtime Component references,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
runtime_ecu_resolution,PASS,0,0,Runtime ECU references,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
runtime_id_uniqueness,PASS,0,0,Canonical RuntimeEvent IDs must be unique,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
runtime_incident_linkage,PASS,0,0,Runtime records with execution IDs should link to derived Incidents,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z
runtime_row_preservation,PASS,38548484,38548484,Silver valid runtime rows must be preserved,8573dcf32f2ed486,medium,v1,7adc84c0-3483-4dd8-acf5-d1c4859c3363,2026-10-14T06:46:22.484Z


In [0]:
# ─────────────────────────────────────────────────
# MEASUREMENT LINKAGE VERIFICATION
# Measurements remain in Silver only — no canonical
# Measurement entities, no graph nodes.
# Verify execution_ref resolves to canonical TestExecutions.
# ─────────────────────────────────────────────────

measurements_silver_df = (
    spark.table("ontologysim.silver.measurements")
    .filter(F.col("_generation_id") == generation_id)
)

canonical_executions_df = (
    spark.table("ontologysim.canonical.test_executions")
    .filter(F.col("_generation_id") == generation_id)
    .select(
        F.col("source_execution_id").alias("exec_source_id"),
        F.col("canonical_id").alias("exec_canonical_id"),
    )
)

total_measurements = measurements_silver_df.count()

measurements_with_ref = (
    measurements_silver_df
    .filter(
        F.col("execution_ref").isNotNull()
        & (F.trim(F.col("execution_ref")) != "")
    )
)

rows_with_ref = measurements_with_ref.count()

distinct_refs = (
    measurements_with_ref
    .select(F.trim(F.col("execution_ref")).alias("execution_ref"))
    .distinct()
    .count()
)

# Check resolution against canonical TestExecutions
unresolved_refs = (
    measurements_with_ref
    .select(F.trim(F.col("execution_ref")).alias("execution_ref"))
    .distinct()
    .alias("m")
    .join(
        canonical_executions_df.alias("e"),
        F.col("m.execution_ref") == F.col("e.exec_source_id"),
        "left",
    )
    .filter(F.col("e.exec_canonical_id").isNull())
    .count()
)

print("═" * 60)
print("MEASUREMENT LINKAGE VERIFICATION")
print("═" * 60)
print(f"Total measurement rows:           {total_measurements}")
print(f"Rows with execution_ref:          {rows_with_ref}")
print(f"Distinct execution_ref:           {distinct_refs}")
print(f"Unresolved execution_ref:         {unresolved_refs}")
print()
if unresolved_refs:
    print("WARNING: Some execution_ref values do not resolve to canonical TestExecutions")
else:
    print("All execution_ref values resolve to canonical TestExecutions")
print()
print("Measurements remain in Silver as Databricks evidence — no canonical entities or graph nodes created.")

════════════════════════════════════════════════════════════
MEASUREMENT LINKAGE VERIFICATION
════════════════════════════════════════════════════════════
Total measurement rows:           7603360
Rows with execution_ref:          7603360
Distinct execution_ref:           160000
Unresolved execution_ref:         0

All execution_ref values resolve to canonical TestExecutions

Measurements remain in Silver as Databricks evidence — no canonical entities or graph nodes created.


In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS ontologysim.graph_ready"
)

print("graph_ready schema ready")

graph_ready schema ready


In [0]:
def build_graph_nodes(
    table_name,
    node_type,
    display_column,
    property_columns,
):
    df = (
        spark.table(table_name)
        .filter(
            F.col("_generation_id")
            == generation_id
        )
    )

    existing_properties = [
        column
        for column in property_columns
        if column in df.columns
    ]

    return (
        df.select(
            F.col("canonical_id").alias(
                "node_id"
            ),

            F.lit(node_type).alias(
                "node_type"
            ),

            "canonical_id",

            F.coalesce(
                F.col(display_column)
                .cast("string"),
                F.col("canonical_id"),
            ).alias("display_name"),

            F.to_json(
                F.struct(
                    *[
                        F.col(column).alias(column)
                        for column
                        in existing_properties
                    ]
                )
            ).alias("properties"),

            "mapping_version",

            "_generation_id",
        )
    )

In [0]:
ecu_nodes_df = build_graph_nodes(
    "ontologysim.canonical.ecus",
    "ECU",
    "ecu_name",
    [
        "source_ecu_id",
        "role",
        "authoritative_source",
    ],
)

component_nodes_df = build_graph_nodes(
    "ontologysim.canonical.components",
    "COMPONENT",
    "component_name",
    [
        "source_component_id",
        "component_type",
        "canonical_ecu_id",
        "responsibility",
        "version",
        "status",
        "owner_team",
    ],
)

requirement_nodes_df = build_graph_nodes(
    "ontologysim.canonical.requirements",
    "REQUIREMENT",
    "display_name",
    [
        "source_requirement_id",
        "requirement_type",
        "priority",
        "status",
    ],
)

test_case_nodes_df = build_graph_nodes(
    "ontologysim.canonical.test_cases",
    "TEST_CASE",
    "display_name",
    [
        "source_test_case_id",
        "test_type",
        "priority",
        "status",
    ],
)

test_execution_nodes_df = build_graph_nodes(
    "ontologysim.canonical.test_executions",
    "TEST_EXECUTION",
    "source_execution_id",
    [
        "source_execution_id",
        "source_test_case_id",
        "result",
        "vehicle_id",
    ],
)

defect_nodes_df = build_graph_nodes(
    "ontologysim.canonical.defects",
    "DEFECT",
    "display_name",
    [
        "source_defect_id",
        "severity",
        "priority",
        "status",
    ],
)

incident_nodes_df = build_graph_nodes(
    "ontologysim.canonical.incidents",
    "INCIDENT",
    "source_execution_id",
    [
        "source_execution_id",
        "canonical_test_execution_id",
        "first_event_at",
        "last_event_at",
        "runtime_event_count",
        "runtime_sources",
        "session_ids",
        "derivation_method",
    ],
)


graph_nodes_df = (
    ecu_nodes_df
    .unionByName(component_nodes_df)
    .unionByName(requirement_nodes_df)
    .unionByName(test_case_nodes_df)
    .unionByName(test_execution_nodes_df)
    .unionByName(defect_nodes_df)
    .unionByName(incident_nodes_df)
)


write_generation(
    graph_nodes_df,
    "ontologysim.graph_ready.nodes",
)

print(
    "Graph-ready nodes:",
    graph_nodes_df.count(),
)

Graph-ready nodes: 348803


In [0]:
canonical_relationships_df = (
    spark.table(
        "ontologysim.canonical.relationships"
    )
    .filter(
        F.col("_generation_id")
        == generation_id
    )
    .filter(
        F.col("relationship_status")
        == "RESOLVED"
    )
)


graph_edges_df = (
    canonical_relationships_df
    .select(
        F.col("relationship_id").alias(
            "edge_id"
        ),

        F.col("source_canonical_id").alias(
            "source_node_id"
        ),

        "relationship_type",

        F.col("target_canonical_id").alias(
            "target_node_id"
        ),

        "relationship_origin",
        "relationship_status",
        "source_family",
        "mapping_version",

        F.to_json(
            F.struct(
                "mapping_method",
                "mapping_confidence",
                "source_record_id",
                "target_source_reference",
            )
        ).alias("properties"),

        "_generation_id",
    )
)


write_generation(
    graph_edges_df,
    "ontologysim.graph_ready.edges",
)

print(
    "Graph-ready edges:",
    graph_edges_df.count(),
)

Graph-ready edges: 379907


In [0]:
nodes_current_df = (
    spark.table(
        "ontologysim.graph_ready.nodes"
    )
    .filter(
        F.col("_generation_id")
        == generation_id
    )
)

edges_current_df = (
    spark.table(
        "ontologysim.graph_ready.edges"
    )
    .filter(
        F.col("_generation_id")
        == generation_id
    )
)


duplicate_node_ids = (
    nodes_current_df
    .groupBy("node_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_edge_ids = (
    edges_current_df
    .groupBy("edge_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


node_lookup_df = (
    nodes_current_df
    .select(
        F.col("node_id").alias("_node_id")
    )
)


missing_source_nodes = (
    edges_current_df.alias("edge")
    .join(
        node_lookup_df.alias("node"),
        F.col("edge.source_node_id")
        == F.col("node._node_id"),
        "left_anti",
    )
    .count()
)


missing_target_nodes = (
    edges_current_df.alias("edge")
    .join(
        node_lookup_df.alias("node"),
        F.col("edge.target_node_id")
        == F.col("node._node_id"),
        "left_anti",
    )
    .count()
)


missing_node_versions = (
    nodes_current_df
    .filter(
        F.col("mapping_version").isNull()
    )
    .count()
)


missing_edge_versions = (
    edges_current_df
    .filter(
        F.col("mapping_version").isNull()
    )
    .count()
)


print(
    "Nodes:",
    nodes_current_df.count(),
)

print(
    "Edges:",
    edges_current_df.count(),
)

print(
    "Duplicate node IDs:",
    duplicate_node_ids,
)

print(
    "Duplicate edge IDs:",
    duplicate_edge_ids,
)

print(
    "Missing source nodes:",
    missing_source_nodes,
)

print(
    "Missing target nodes:",
    missing_target_nodes,
)

print(
    "Missing node mapping versions:",
    missing_node_versions,
)

print(
    "Missing edge mapping versions:",
    missing_edge_versions,
)


if duplicate_node_ids:
    raise ValueError(
        "Duplicate graph node IDs"
    )

if duplicate_edge_ids:
    raise ValueError(
        "Duplicate graph edge IDs"
    )

if missing_source_nodes:
    raise ValueError(
        "Graph edges reference missing source nodes"
    )

if missing_target_nodes:
    raise ValueError(
        "Graph edges reference missing target nodes"
    )

if missing_node_versions:
    raise ValueError(
        "Graph nodes missing mapping_version"
    )

if missing_edge_versions:
    raise ValueError(
        "Graph edges missing mapping_version"
    )

print("Graph-ready validation passed")

Nodes: 348803
Edges: 379907
Duplicate node IDs: 0
Duplicate edge IDs: 0
Missing source nodes: 0
Missing target nodes: 0
Missing node mapping versions: 0
Missing edge mapping versions: 0
Graph-ready validation passed


In [0]:
print("NODE TYPES")

display(
    nodes_current_df
    .groupBy("node_type")
    .count()
    .orderBy("node_type")
)


print("EDGE TYPES")

display(
    edges_current_df
    .groupBy("relationship_type")
    .count()
    .orderBy("relationship_type")
)

NODE TYPES


node_type,count
COMPONENT,1600
DEFECT,3200
ECU,3
INCIDENT,160000
REQUIREMENT,8000
TEST_CASE,16000
TEST_EXECUTION,160000


EDGE TYPES


relationship_type,count
AFFECTS,6400
EXECUTION_OF,155200
FOUND_DURING,2947
HOSTS,1600
INVOLVES,160000
RELATES_TO,6400
TARGETS,47360
